# Orbit Wars -- v8 A100 run (mixed 2p/4p league, dual Elo, invited members)

**v8 = v7 (BC warm-start -> guardrailed PPO, categorical WHERE) + the 4-player program.**
The it1041 post-mortem: a checkpoint can DOMINATE heads-up (1.000 gauntlet) and still LOSE
4-player FFAs (29% top-rate vs it101's 58%) -- and the Kaggle leaderboard mixes both formats.
v8 trains both formats inside one league:

- **Mixed 2p/4p training.** Every iteration is a 2-player match OR a 4-player FFA in the same
  GpuEnv (N-player combat = the official top-vs-second resolve; official 4p seating: owners 0..3
  on one symmetric home group). The learner is always player 0; FFA seats 1-3 are PFSP-sampled
  league members. Neural seats act per step (no-grad); scripted anchors run fully on-GPU.
  The 2p path is bit-exact to v7 (`scripts/test_v8_parity.py`).
- **Dual correlated Elo.** The learner and every member carry a 2p rating AND a 4p rating.
  A 2p result updates elo2 (and drags elo4 by `ELO_COUPLING`); a 4p FFA becomes 3 pairwise
  updates at `ELO_K4/3` each (coupled back into elo2). Scripted + invited members are ANCHORED
  in both formats.
- **Gap-driven match mix, bounded 1:1 .. 1:6.** `share_4p = clip(S4_BASE + S4_GAIN*(elo2-elo4)/400,
  S4_MIN, S4_MAX)`. When the two ratings split apart, the lagging format gets more matches; the
  2p:4p ratio may tilt toward 4p but never beyond 1:6, and never below 1:1. The scheduler is
  error-diffused, so the REALIZED ratio equals the target (not just in expectation).
- **Invited league members.** The first existing folder in `INVITE_DIRS` is scanned for `*.pt`
  checkpoints (pre-trained smaller models; dims come from each ckpt's embedded config, else
  `INVITE_CFG` -- all invited members share one dimension). The TOP-4 by anchored Elo join as
  high-level bots; the anchor comes from the filename suffix `_elo<NNNN>` (e.g. `it1_elo1100.pt`),
  else ckpt meta `elo`, else `INVITE_DEFAULT_ELO`, and is NEVER updated. They sample at
  `INVITE_MATCH_BOOST`x while unmastered; once the learner's per-format score EMA passes
  `INVITE_MASTER_WR`, their matchup rate DROPS to `INVITE_MASTERED_W`x (anti-forgetting floor).
  High-level sparring -- especially 4p -- without spending compute training those models.
- **Weak-point matchmaking.** Per member and per format, the league compares the learner's ACTUAL
  score EMA against the Elo-EXPECTED score. When reality runs below theory (Elo says 0.90, the
  learner only scores 0.70), that member's matchup rate is boosted by
  `1 + WEAK_BOOST_GAIN*(expected - actual)` capped at `WEAK_BOOST_MAX` (x3 at the example gap).
  It ONLY targets weakness: overperforming opponents are never down-weighted, and the detector
  stays off until `WEAK_BOOST_MIN_GAMES` matches / outside the EMA-noise deadband. Active boosts
  show on the leaderboard as `weak2/weak4 x<mult>`.
- **Mixed best-ckpt gauntlet.** best = `(1-GAUNTLET_4P_W)`*2p-gauntlet + `GAUNTLET_4P_W`*4p-gauntlet
  (FFA vs the starter/greedy/intermediate trio), so 2p-only specialists no longer win the
  checkpoint race.

**Colab checklist** (unchanged from v7)
1. Runtime -> A100 GPU. Optionally `import os; os.environ['OW_CKPT_DIR']='/content/drive/MyDrive/ow_v8'`
   BEFORE the config cell (and `os.environ['OW_LEAGUE_DIR']=...` to point at the invited-members
   folder). Resume by setting `RESUME_FROM = TRAIN_STATE_PATH` and re-running the run cell.
2. Watch the first ~15 heartbeats: it1-5 must show `kl 0.000` (critic-only warmup), then KL in the
   0.02-0.05 band. The heartbeat now shows the format (`2p`/`4p`), both Elos and the 4p share.
3. The `[best]` line prints the MIXED gauntlet (2p / 4p breakdown alongside) -- that is the number
   that matters. Healthy run: both gauntlets rising, launch-rate roughly flat near its BC level,
   no `[TRIPWIRE]` lines, and `s4` drifting toward whichever format lags.


## 1. Imports
Only `torch`, `numpy`, `matplotlib` -- all preinstalled on Colab / Kaggle.

In [ ]:
# Colab/Kaggle usually ship torch already. Uncomment to (re)install a CUDA build if needed.
# import sys, subprocess
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'numpy', 'matplotlib'])
try:
    import torch  # noqa: F401
    print('torch present')
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'numpy', 'matplotlib'])

In [ ]:
import math
import random
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.set_float32_matmul_precision("high")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32  # integer-in-float32 convention (mirrors the native env)
print("device:", DEVICE, "| torch", torch.__version__)

## 2. Config (editable hyperparameters)

Mirrors scripts/run_setup1_target.cmd (v5 target actor). Set SMOKE = True for a fast
Colab/Kaggle sanity run (tiny model, fewer iters, smaller batch); SMOKE = False (default) is
the full spec.

Full run uses B=128 (num_groups 16 * group_size 8), width 512, 6 res blocks. You can drop
B/HIDDEN to fit VRAM. PPO has no groups (the critic is the baseline), so B is just the env count.

In [ ]:
# ============================================================================
# CONFIG  --  set SMOKE=False for the real run (the delivered default).
# ============================================================================
# A SINGLE boolean controls everything: SMOKE=True shrinks the model/batch/iters
# for a fast end-to-end sanity run; SMOKE=False is the full docs/set-ups/1.md v5
# target-actor spec (matches scripts/run_setup1_target.cmd).
SMOKE = False   # <<< FLIP to True for a quick Colab/Kaggle smoke; False = full run

# ---- model size (full spec: width 512, 6 residual blocks) -------------------
HIDDEN          = 64  if SMOKE else 768    # d: trunk width  ("model dim")
N_RES_BLOCKS    = 2   if SMOKE else 6      # residual MLP blocks in the TRUNK (pre-LN ResNet)
# ---- trunk architecture (configurable: stacked transformer w/ ResNet-MLP ends) ----
# CPU-probe sizing: HIDDEN=256 + N_TX_LAYERS=4 ~ same fwd cost as the old h=512 trunk; h=512,L=4 ~3x.
ARCH         = "transformer"  # "trunk" = legacy (1x attn + N res-MLP) | "transformer" = N-layer stack
N_TX_LAYERS  = 6              # [transformer] encoder layers, each = multi-head attn + ResNet-MLP
N_HEADS      = 12             # [transformer] attention heads (HIDDEN must be divisible by this)
TX_MLP_RATIO = 4              # [transformer] per-layer MLP hidden = ratio x HIDDEN
N_STEM_RES   = 1              # [transformer] ResNet-MLP blocks BEFORE the stack (input end)
N_HEAD_RES   = 4              # [transformer] ResNet-MLP blocks AFTER  the stack (output end)
VALUE_RES_BLOCKS = 1  if SMOKE else 2      # residual MLP blocks in the VALUE/critic head (same ResNet)
D_G             = 32                        # board-globals embedding dim
USE_GLU         = True
USE_ATTENTION   = True                      # cross-planet self-attention block (A/B: False = ablate it)

# ---- env / batch (B = NUM_GROUPS * GROUP_SIZE; PPO has no groups, B = env count)
NUM_GROUPS      = 4   if SMOKE else 16
GROUP_SIZE      = 4   if SMOKE else 16
B               = NUM_GROUPS * GROUP_SIZE   # parallel envs (= 16 smoke / 128 full)
EPISODE_STEPS   = 200 if SMOKE else 500
PLANET_CAP      = 48                         # E: planet slots/env == dest-categorical size (comets omitted)
FLEET_CAP       = 512 if SMOKE else 1024     # max simultaneous in-flight fleets/env
SHIP_SPEED      = 6.0
COMETS_ENABLED  = True
COMET_OFFICIAL  = True                       # official elliptical waypoint comets (CPU world-gen precompute); False = legacy straight-line pair
COMET_MAX_LEN   = 40                         # official visible-path cap (5..40 waypoints)                       # spawn comets in training (hidden schedule, like the official engine)
COMET_SPAWN_STEPS = (50, 150, 250, 350, 450) # ticks at which a symmetric comet pair appears
COMET_SPEED     = 4.0                         # comet straight-line speed (official cometSpeed)
COMET_PAIRS     = 1                           # point-symmetric comet pairs per spawn (-> 2 comets)
COMET_SPAWN_RADIUS = 50.0                     # radius from center where comets enter
COMET_PERI_MIN  = 15.0                        # comet closest-approach to center (min)
COMET_PERI_MAX  = 30.0                        # comet closest-approach to center (max)
COMET_EXPIRE_RADIUS = 56.0                    # comet removed (ships lost) once it sweeps back past this

# ---- PPO + GAE --------------------------------------------------------------
LR              = 1e-4
GAMMA           = 0.997
GAE_LAMBDA      = 0.98
CLIP            = 0.2
VF_COEF         = 0.5
ENT_COEF        = 0.005
MINIBATCHES     = 16  if SMOKE else 64
UPDATE_EPOCHS   = 3
MAX_GRAD_NORM   = 0.5
KL_TARGET       = 0.05  # PPO guardrail: stop the update epoch once a minibatch's KL exceeds this (anti-runaway)
LOGRATIO_CLAMP  = 4.0   # PPO guardrail: clamp log importance-ratio before exp (ratio in [e^-4, e^4]; no overflow)
LOGIT_CLAMP     = 8.0   # Dirichlet guardrail: clamp dest_logits before softplus so alpha can't explode
ADAM_EPS        = 1e-5

# ---- training length (NO hand-coded stages -- the Elo/PFSP league IS the curriculum) ----
# Every iter the opponent is PFSP-sampled from the league {random, starter, snapshots} by Elo,
# so the schedule EMERGES from skill: a fresh learner (rated at the random anchor) plays the weak
# scripted bots, and as its rating climbs it shifts to ever-stronger historical snapshots. The old
# stage1/2/3 boundaries are gone -- the Elo ladder reproduces them organically.
TOTAL_ITERS       = 4 if SMOKE else 600
SELFPLAY_REFRESH  = 5 if SMOKE else 25   # snapshot the learner into the pool every N iters
SNAPSHOT_WARMUP   = 0   # BC init is already strong -> snapshot from the start   # anchors-only warmup: no learner snapshot before this iter

# ---- self-play Elo LEAGUE (PFSP pool -- the SOLE opponent source, no stages) -----
# EVERY iter the opponent is PFSP-sampled from ALL pool members (random + starter anchors +
# learner snapshots), weighted by f(p)+floor where p is the Elo-expected score. The two scripted
# bots are rating-ANCHORED (fixed Elo) so the scale -- and the auto-curriculum -- are pinned.
LEAGUE_MAX_SNAPSHOTS = 4 if SMOKE else 16   # max AUTO learner snapshots kept (FIFO; pinned ckpts exempt)
LEAGUE_INIT_CKPTS    = []     # paths to past .pt weights to seed the pool (pinned, never pruned)
ELO_INIT             = 0.0    # learner starts at the random anchor; climbs the ladder as it improves
ELO_RANDOM           = 0.0    # ANCHORED (fixed) rating of the random bot  -> pins the Elo scale
ELO_STARTER          = 450.0  # ANCHORED rating of the starter bot (Kaggle-leaderboard-calibrated: submitted starter ~450)
ELO_MEDIUM           = 500.0  # MEDIUM bot (advanced; locked until the advance gate; submitted medium ~500 on the leaderboard)
ELO_GREEDY           = 500.0   # GREEDY local-capture bot (advanced; locked until the advance gate; submitted greedy ~500)
ELO_INTERMEDIATE     = 475.0   # INTERMEDIATE bot (never submitted: interpolated between starter 450 and greedy 500, keeps the in-house ordering); ALWAYS-ON, not gated
ADVANCE_TRIGGER_ELO  = 525.0   # learner ELO that triggers an all-anchor recalibration to consider unlocking (+50 over intermediate, as before the rescale)
ADVANCE_CONFIRM_ELO  = 425.0   # if the all-anchor recal puts the learner below this, stay in base regime (anti-inflation; -50 under intermediate)
ELO_K                = 32.0   # Elo K-factor (bumped: Elo now DRIVES sampling, so track skill fast)
PFSP_MODE            = "even" # snapshot sampling: "even" (p*(1-p)) / "hard" ((1-p)^P) / "uniform"
PFSP_POWER           = 2.0    # exponent when PFSP_MODE == "hard"
PFSP_FLOOR           = 0.05   # uniform floor added to every snapshot weight (anti-forgetting)
MATCH_ELO_W          = 0.7    # matchmaking weight on Elo (evenly-matched opponents)
MATCH_FP_W           = 0.3    # matchmaking weight on behavioral-footprint distance (anti-clone)
SCRIPT_MATCH_BOOST   = 2.0    # x weight on UNMASTERED scripted anchors (2.0 = +100%); spend compute on the bots until beaten
SCRIPT_BOOST_DROP_WR = 0.90   # drop a scripted anchor's boost once learner EMA win-rate vs it >= this
SCRIPT_BOOST_MIN_GAMES = 5    # min matches vs an anchor before its boost can be dropped
WR_EMA_BETA          = 0.1    # EMA smoothing for per-opponent learner win-rate (responsive 90% gate)
CAPTURE_RADIUS       = 25.0   # greedy bot: only captures non-owned planets within this distance
CAPTURE_MARGIN       = 2.0    # greedy/intermediate: ships sent = floor(target_ships) + this (transit-production buffer)
REBALANCE_PCT        = 0.4    # intermediate: rebalance when a source's surplus over the local-poorest > this fraction of garrison
REBALANCE_RADIUS     = 30.0   # intermediate: only rebalance to owned planets within this distance (local, efficient)

# ---- gated Dirichlet allocation action space + sun-reachability -------------
REACH_MASK   = True           # hard-mask destinations the sun would absorb (exact for straight shots)
LEAD_TARGET  = True           # aim at the orbiting destination's INTERCEPT, not its current position
# ---- gated Dirichlet allocation: per-source WHERE simplex (48x48) x launch GATE -----
WHERE_DIST       = 'categorical'  # 'categorical' (v7 default: bounded log-prob, search-ready) | 'dirichlet' (v6 A/B)
ALLOC_KAPPA      = 1.0   # Dirichlet concentration scale: alpha = softplus(logits)*kappa + eps
MIN_LAUNCH_SHIPS = 2     # drop launches below this many ships (-> stay home; no 1-ship dribbles)
GATE_TRIM_LO     = 0.2   # gate fire-prob trim: sigmoid(gate)<=this -> never fire (deadzone)
GATE_TRIM_HI     = 0.8   # gate fire-prob trim: sigmoid(gate)>=this -> always fire (saturate)
GATE_EPS         = 0.02  # TRAINING: keep fire-prob in [eps,1-eps] (no freeze); DEPLOY greedy fires hard 0/1
INIT_GATE_BIAS   = -1.0  # gate-head bias init: start cautious (few planets fire at init)
POLICY_MODE      = "ppo" # "ppo" = gated-Dirichlet PPO | "mcts" = discretized tree search (stub)
LAUNCH_REWARD    = 0.001 # dispatch reward: factor on ships launched to NON-ego planets (attack/expand), per step. Tune vs production (mean ~2.7/planet, ~76/world)
SELF_LAUNCH_REWARD = 0.001 # dispatch reward: factor on ships launched to EGO-owned planets (reinforce), per step; tunable (<= LAUNCH_REWARD to favor attacks)
SELF_LAUNCH_CAP  = 10.0  # CAP 1: max EGO-dest ships/step counted in the dispatch reward (non-ego launches UNCAPPED)
LAUNCH_STEP_CAP  = 0.5   # CAP 2: overall per-step cap on the dispatch reward (enemy*f + capped self*f, combined)
LAUNCH_WINDOW    = 150   # launch reward VALID only for the first N steps; a WIN before step N pays the unspent window IN FULL (anti-stall cashout)
LAUNCH_GAME_CAP  = 30.0  # CAP 3: cap on TOTAL launch reward over the whole game (per-step + cashout); window-max = LAUNCH_STEP_CAP*LAUNCH_WINDOW

# ---- reward (v5, docs/set-ups/1.md -- REPLACES the old event reward) --------
WIN_BONUS         = 1000.0   # win outcome
LOSS_PENALTY      = 1000.0   # loss outcome (symmetric)
WIN_DECAY         = 1.0    # win value  = +WIN_BONUS   * WIN_DECAY^max(0, len - DECAY_START)
LOSS_DECAY        = 1.0    # loss value = -LOSS_PENALTY* LOSS_DECAY^max(0, len - DECAY_START)
DECAY_START_STEP  = 100.0    # win/loss FLAT for the first 100 steps, then decay
CAPTURE_REWARD    = 30.0     # + per planet gained (per ego owner-flip, per step)
CAPTURE_LOSS_FRAC = 0.9      # losing a planet removes only this fraction of CAPTURE_REWARD (net +10%/capture)
CAPTURE_PROD_SCALE = 0.2     # per-planet capture reward = CAPTURE_REWARD * (1 + this * production); higher = grab valuable planets
PROD_MILESTONE_REWARD = 20.0 # +20 each time ego TOTAL ships crosses a doubling threshold
PROD_MILESTONE_BASE   = 100.0
PPO_REWARD_SCALE  = 100.0    # divide per-step reward AND outcome by this before GAE (value stability)

# ======================= v6 additions =======================
# ---- compute-bound on A100 (bf16 autocast + flash SDPA + fused Adam + on-GPU rollout) ----
USE_AMP   = True             # bf16 autocast on the net forward (dist log-prob/entropy/value stay fp32)
AMP_DTYPE = torch.bfloat16   # bf16: full fp32 range -> NO GradScaler, safe vs the Dirichlet log-prob
# ---- action gate: sharpened greedy WHERE (gate already handles fire/hold) + dest calm-init ----
GREEDY_TAU     = 0.5         # temperature for the sharpened-softmax greedy WHERE (concentrated strike)
INIT_DEST_SCALE = 0.02       # small-init the WHERE head -> ~uniform allocation at start
DEST_HEAD      = 'bilinear'  # 'linear' (per-source) | 'bilinear' (query=src, key=dst pointer score)
# ---- entropy schedule (anti collapse) ----
ENT_DIR_COEF   = 0.0         # weight of the Dirichlet term inside dist.entropy() (0 = gate-only bonus)
ENT_COEF_START = 0.005
ENT_COEF_END   = 0.002
ENT_DECAY_ITERS = 300
CUR_ENT_COEF   = ENT_COEF_START   # mutated each iter by anneal_ent_coef()
# ---- PPO KL early-stop (end-of-epoch, mean over the epoch) ----
KL_EARLYSTOP = True
# ---- value normalization (PopArt: output-preserving running return std) ----
USE_POPART  = True
POPART_BETA = 3e-4
# ---- potential-based reward shaping (Ng et al. 1999; policy-invariant, un-hackable) ----
# Replaces the farmable capture/milestone/launch channels with F = gamma*Phi(s') - Phi(s):
#   Phi_ship = shiplog(my_ships) - shiplog(enemy_ships);  Phi_prod = (my_prod - enemy_prod)/total_prod.
USE_POTENTIAL_SHAPING = True
SHAPE_SHIP = 25.0
SHAPE_PROD = 25.0

# ---- checkpointing (Colab/Kaggle-friendly path) -----------------------------
import os
CKPT_DIR = os.environ.get("OW_CKPT_DIR", "/kaggle/working" if os.path.isdir("/kaggle/working")
                          else "/content" if os.path.isdir("/content") else ".")
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT_PATH = os.path.join(CKPT_DIR, "setup1_target_policy.pt")
BEST_CKPT_PATH = os.path.join(CKPT_DIR, "setup1_target_policy_best.pt")
TRAIN_STATE_PATH = os.path.join(CKPT_DIR, "setup1_target_train_state.pt")  # full resumable state
CKPT_EVERY = 20   # fixed-opponent gauntlet + best-ckpt cadence (dense: catch peaks/ratchets early)  # save cadence (iters); best-by-win-rate is saved separately

# ---- BC warm-start + PPO stability guardrails (v7) --------------------------
BC_ENABLED        = True   # Run-All: BC pretrain -> PPO warm-start  # run bc_pretrain() before training (or pass resume_from=BC_CKPT_PATH)
BC_ROUNDS         = 12     # BC data rounds (1 round = one B-env rollout of the clone policy)
BC_EPOCHS         = 2      # supervised epochs per round (fresh on-policy expert data each round)
BC_LR             = 3e-4   # Adam LR for the supervised phase
BC_MINIBATCH      = 4096   # source-planet samples per BC minibatch (flattened over B*T*E)
BC_CKPT_PATH      = os.path.join(CKPT_DIR, "bc_medium_init.pt")
VALUE_WARMUP_ITERS = 5     # first N iters of a train() call: policy+entropy frozen, critic+PopArt only
LR_WARMUP_ITERS    = 8     # linear LR warmup over the first N iters of a train() call (0 = off)
KL_STOP_MINIBATCH  = True
TRIPWIRE_FRAC      = 0.4   # warn when launch-rate EMA < this frac of the early baseline (passivity-ratchet detector)
TRIPWIRE_BASE_ITERS = 10   # iters of this train() call that define the launch-rate baseline  # break the PPO update MID-EPOCH once a minibatch KL > KL_HARD_MULT*KL_TARGET
KL_HARD_MULT       = 1.5

# ---- world pool -------------------------------------------------------------
N_WORLDS        = 256 if SMOKE else 2048
SEED            = 0
WORLD_RESAMPLE_EVERY = 0 if SMOKE else 100    # regenerate the world pool every N GLOBAL iters (0 = fixed pool)
ELO_RECAL_EVERY      = 0 if SMOKE else 500   # re-ground league ELO vs anchors every N GLOBAL iters (decoupled from reseed)
RESUME_FROM          = None                    # set to TRAIN_STATE_PATH (or a path) to resume + train more
ELO_RECAL_ENVS       = 64                      # games vs each fixed anchor when re-grounding ELO at a resample boundary

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("SMOKE =", SMOKE, "| B =", B, "| HIDDEN =", HIDDEN,
      "| N_RES_BLOCKS =", N_RES_BLOCKS, "| TOTAL_ITERS =", TOTAL_ITERS)
print("checkpoints ->", CKPT_PATH)
# ======================= v8 additions: mixed 2p/4p + dual Elo + invited league =======================
# ---- 4-player FFA training (player 0 = learner; seats 1..3 PFSP-sampled from the league) ----
FOURP_ENABLED   = True
B_4P            = B            # parallel envs per 4p iteration (learner buffers match 2p; each
                               # NEURAL opponent seat adds one no-grad forward per step)
OUTCOME_4P      = "placement"  # "placement": outcome = 2*mean(pairwise score) - 1 (placement-linear,
                               #   beat-all=+1 .. lose-all=-1; denser learning signal)
                               # "winner": official top-score-only (+1 top, -1 otherwise)
# ---- match-type scheduler: 2p vs 4p share driven by the DUAL-Elo gap ----
# share_4p = clip(S4_BASE + S4_GAIN*(elo2p - elo4p)/400, S4_MIN, S4_MAX). When the 4p rating falls
# behind the 2p rating the system schedules more 4p (and vice versa). Bounds = the 1:1 .. 1:6
# envelope: MORE 4p than 2p is allowed, but never beyond 6 4p-iters per 2p-iter and never fewer 4p
# than 2p. The train loop error-diffuses the share, so the realized ratio is exact.
S4_BASE         = 0.60         # baseline share of 4p iterations (0.60 = 2p:4p of 2:3)
S4_MIN          = 0.50         # lower bound  = 1:1 (2p:4p)
S4_MAX          = 6.0 / 7.0    # upper bound  = 1:6 (2p:4p)
S4_GAIN         = 1.0          # share shift per 400 Elo of (elo2p - elo4p) gap
# ---- dual correlated Elo (2p Elo + 4p Elo) ----
ELO_K4          = 32.0         # 4p K-factor; each of the 3 pairwise updates uses ELO_K4/3
ELO_COUPLING    = 0.25         # cross-format fraction: a 2p delta also moves the 4p rating by this
                               # much (and vice versa) -- the two ratings stay correlated
# ---- invited league members (pre-trained smaller models, treated as anchored high-level bots) ----
# The FIRST existing directory below is scanned for *.pt checkpoints and the TOP-INVITE_MAX by
# anchored Elo are invited. The anchor comes from the filename suffix `_elo<NNNN>` (e.g.
# it1_elo1100.pt), else the ckpt meta "elo", else INVITE_DEFAULT_ELO. Invited ratings are MANUALLY
# ANCHORED: they never drift. NOTE: CKPT_DIR/league_agents is the AUTO-SNAPSHOT dump of this run --
# deliberately NOT searched (use CKPT_DIR/league_invited or OW_LEAGUE_DIR for curated members).
INVITE_DIRS     = [os.environ.get("OW_LEAGUE_DIR", ""),
                   "/content/drive/MyDrive/league_agents",
                   os.path.join(CKPT_DIR, "league_invited"),
                   "league_agents",
                   os.path.join("notebooks", "league_agents")]
INVITE_MAX         = 4         # invite at most the top-4 members from the folder
INVITE_DEFAULT_ELO = 750.0     # anchored Elo when neither filename nor ckpt meta carries one
                               # (= the submitted it1041 weight's leaderboard level)
INVITE_ELO4_OFFSET = 0.0       # anchored 4p Elo = anchored 2p Elo + this offset
INVITE_MATCH_BOOST = 2.0       # x sampling weight while UNMASTERED (high-level sparring focus)
INVITE_MASTER_WR   = 0.75      # mastered once the learner's per-format score EMA >= this ...
INVITE_MASTERED_W  = 0.2       # ... then the matchup rate DROPS to this x (>0: anti-forgetting)
# fallback dims when an invited ckpt has no embedded config (all invited members share one dim)
INVITE_CFG = {"HIDDEN": 256, "N_TX_LAYERS": 4, "N_HEADS": 8, "TX_MLP_RATIO": 4,
              "N_STEM_RES": 1, "N_HEAD_RES": 1, "N_RES_BLOCKS": 6, "VALUE_RES_BLOCKS": 2,
              "ARCH": "transformer", "USE_GLU": True, "USE_ATTENTION": True, "D_G": 32}
# ---- weak-point matchmaking (per format) ----
# If the learner's ACTUAL score EMA vs a member runs below the Elo-EXPECTED score, that member is
# a weak point -> boost its matchup rate by 1 + WEAK_BOOST_GAIN*(expected - actual), capped at
# WEAK_BOOST_MAX. Example: Elo says 0.90 but reality is 0.70 -> gap 0.20 -> x3. Never fires the
# other way (overperforming opponents are NOT down-weighted) and never below the deadband.
WEAK_BOOST_ENABLED   = True
WEAK_BOOST_MIN_GAMES = 5       # per-format matches vs a member before the detector can fire
WEAK_BOOST_MIN_GAP   = 0.05    # deadband: ignore (expected - actual) gaps below this (EMA noise)
WEAK_BOOST_GAIN      = 10.0    # boost = 1 + gain * gap  (0.90 expected vs 0.70 actual -> x3)
WEAK_BOOST_MAX       = 3.0     # cap on the multiplier
# ---- mixed best-ckpt gauntlet ----
GAUNTLET_4P_W   = 0.5          # best score = (1-w)*2p gauntlet + w*4p gauntlet (it1041 lesson:
                               # 2p-dominant ckpts can LOSE 4p; the leaderboard mixes both)


## 3. Feature layout (mirrors `core/encode.hpp` + `core/state.hpp`)

`F = 11 body + 90 threat = 101` features/planet; `G = 10` board globals (a separate input).
Board constants are the competition's. The threat features keep the `N_SOON=15` soonest +
`N_BIG=15` largest inbound fleets per planet, 3 feats each.

In [ ]:
# board constants (core/state.hpp)
BOARD_SIZE = 100.0
CENTER = BOARD_SIZE / 2.0
SUN_RADIUS = 10.0
ROTATION_RADIUS_LIMIT = 50.0
COMET_RADIUS = 1.0
COMET_PRODUCTION = 1
PI = math.pi

# feature layout (core/encode.hpp)
N_SOON = 15
N_BIG = 15
N_THREAT_FLEETS = N_SOON + N_BIG            # 30 slots * 3 feats = 90
N_BODY_FEATURES = 11
N_ENTITY_FEATURES = N_BODY_FEATURES + 3 * N_THREAT_FLEETS   # 32
N_GLOBAL_FEATURES = 10
F_DIM = N_ENTITY_FEATURES
G_DIM = N_GLOBAL_FEATURES

SHIP_LOG_DENOM = math.log(1000.0)
DIAG_HALF = math.sqrt(BOARD_SIZE * BOARD_SIZE + BOARD_SIZE * BOARD_SIZE) / 2.0
THREAT_MAX_SPEED = 6.0      # == ship speed; fleet speed cap for the ETA model
THREAT_ETA_SCALE = 100.0
BIG = 1e18

def ship_log_t(x):
    return torch.log1p(x.clamp_min(0.0)) / SHIP_LOG_DENOM

def fleet_speed_t(ships, vmax=SHIP_SPEED):
    # 1 + (vmax-1)*(log(ships)/log(1000))^1.5, capped, ships>=1  (core/encode.hpp)
    n = ships.clamp_min(1.0)
    v = 1.0 + (vmax - 1.0) * torch.pow(torch.log(n) / math.log(1000.0), 1.5)
    return v.clamp_max(vmax)

print("F =", F_DIM, "| G =", G_DIM, "| dest-categorical size E =", PLANET_CAP, "| action params/planet = 2 (dest, phi)")

## 4. Target-actor distribution (mirrors `model/distribution.hpp::TargetActorDist`)
Per OWNED planet: a destination Categorical over the E obs slots (masked to live
planets) x a squashed-Gaussian phi (% of that planet's ships). Joint log-prob / entropy
sum over owned planets only.

In [ ]:
class GatedAllocDist:
    """Gated per-source Dirichlet ALLOCATION actor (v6).
      WHERE  dest_logits (B,E,E) -> alpha = softplus(logits)*kappa + eps -> Dirichlet rows over dests.
      GATE   gate_logits (B,E)   -> p = clamp((sigmoid-LO)/(HI-LO),0,1), trimmed to [eps,1-eps]; fire ~ Bernoulli(p).
    On fire the planet dispatches its FULL garrison routed by its WHERE row; on no-fire it holds.
    Action (B,E,E+1): [...,:E]=allocation rows, [...,E]=fire in {0,1}. Log-prob/entropy SUM over owned sources;
    the WHERE term only counts when firing.
    v6: greedy()/DEPLOY uses a TEMPERATURE-SHARPENED softmax over the LIVE+REACHABLE off-diagonal dest logits
    (concentration-aware) so the deployed policy lands a CONCENTRATED strike -- the Dirichlet MEAN spreads the
    full garrison thin and the decode floor then dropped every launch (the residual greedy-passive bug)."""
    def __init__(self, dest_logits, gate_logits, owned, alive=None, reach=None, ships=None, kappa=ALLOC_KAPPA, deploy=False):
        self.owned = owned if owned.dim() == 2 else owned.squeeze(-1)        # (B,E) legal sources
        logits = dest_logits.clamp(-LOGIT_CLAMP, LOGIT_CLAMP)
        self.alpha = F.softplus(logits) * kappa + 1e-3                       # (B,E,E)
        self.dist = torch.distributions.Dirichlet(self.alpha)
        g = torch.sigmoid(gate_logits if gate_logits.dim() == 2 else gate_logits.squeeze(-1))   # (B,E)
        if deploy:                                                          # DEPLOY: hard deadzone (fire/hold trim)
            p = ((g - GATE_TRIM_LO) / (GATE_TRIM_HI - GATE_TRIM_LO)).clamp(0.0, 1.0)
        else:                                                              # TRAIN: SMOOTH fire-prob so the gate head
            p = g                                                          #   keeps a full-support gradient (the
                                                                           #   deadzone clamp zeroed it -> grad vanish)
        self.p = p.clamp(GATE_EPS, 1.0 - GATE_EPS)                          # (B,E) trainable fire-prob
        E = logits.shape[-1]
        eye = torch.eye(E, dtype=torch.bool, device=logits.device).unsqueeze(0)
        dead = eye.expand_as(logits).clone()                                # drop self (decode drops it)
        if alive is not None:
            dead = dead | (alive < 0.5).unsqueeze(1)                        # drop dead dests
        if reach is not None:
            dead = dead | (reach < 0.5)                                     # drop sun-blocked dests
        self.glogits = logits.masked_fill(dead, -1e9)                       # masked logits for the greedy WHERE
    def _bundle(self, alloc, fire):
        return torch.cat([alloc, fire.unsqueeze(-1)], -1)                   # (B,E,E+1)
    def sample(self):
        return self._bundle(self.dist.rsample(), torch.bernoulli(self.p))  # WHERE simplex + Bernoulli fire
    def greedy(self):
        alloc = torch.softmax(self.glogits / GREEDY_TAU, -1)               # sharpened, mask-aware -> concentrated
        return self._bundle(alloc, (self.p >= 0.5).to(alloc.dtype))        # DEPLOY: deterministic fire
    def log_prob(self, a):
        E = self.alpha.shape[-1]
        alloc = a[..., :E].clamp_min(1e-6); alloc = alloc / alloc.sum(-1, keepdim=True)   # project to simplex
        fire = a[..., E]                                                   # (B,E) in {0,1}
        where_lp = self.dist.log_prob(alloc)                              # (B,E)
        gate_lp = fire * torch.log(self.p.clamp_min(1e-8)) + (1.0 - fire) * torch.log((1.0 - self.p).clamp_min(1e-8))
        lp = gate_lp + fire * where_lp                                    # WHERE only counts when firing
        return (lp * self.owned).sum(1)                                   # (B,)
    def entropy(self):
        p = self.p
        gate_e = -(p * torch.log(p.clamp_min(1e-8)) + (1.0 - p) * torch.log((1.0 - p).clamp_min(1e-8)))
        # Dirichlet differential entropy is unboundedly NEGATIVE when peaked -> as a bonus it
        # blurs a BC-sharpened WHERE head (measured: loss +20, gn 1380, kl 4.8). Gate-only by default.
        ent = gate_e + ENT_DIR_COEF * p * self.dist.entropy()
        return (ent * self.owned).sum(1)                                  # (B,)


class GatedCatDist:
    """Gated per-source CATEGORICAL destination actor (v7 default).
      WHERE  masked logits glogits (B,E,E) -> Categorical over dests; action row = one-hot(dest).
      GATE   identical to GatedAllocDist (smooth p at train, hard trim at deploy).
    Same (B,E,E+1) action bundle as the Dirichlet -> decode/buffers/league untouched.
    log_prob is bounded-sensitivity (log_softmax), which is what makes PPO-after-BC stable."""
    def __init__(self, dest_logits, gate_logits, owned, alive=None, reach=None, ships=None, deploy=False):
        self.owned = owned if owned.dim() == 2 else owned.squeeze(-1)
        logits = dest_logits.clamp(-LOGIT_CLAMP, LOGIT_CLAMP)
        g = torch.sigmoid(gate_logits if gate_logits.dim() == 2 else gate_logits.squeeze(-1))
        if deploy:
            p = ((g - GATE_TRIM_LO) / (GATE_TRIM_HI - GATE_TRIM_LO)).clamp(0.0, 1.0)
        else:
            p = g
        self.p = p.clamp(GATE_EPS, 1.0 - GATE_EPS)
        E = logits.shape[-1]
        eye = torch.eye(E, dtype=torch.bool, device=logits.device).unsqueeze(0)
        dead = eye.expand_as(logits).clone()
        if alive is not None:
            dead = dead | (alive < 0.5).unsqueeze(1)
        if reach is not None:
            dead = dead | (reach < 0.5)
        self.glogits = logits.masked_fill(dead, -1e9)
        self.lsm = torch.log_softmax(self.glogits, -1)                      # (B,E,E)
    def _bundle(self, dest, fire):
        alloc = torch.zeros_like(self.lsm)
        alloc.scatter_(2, dest.unsqueeze(-1), 1.0)                          # one-hot WHERE row
        return torch.cat([alloc, fire.unsqueeze(-1)], -1)
    def sample(self):
        B_, E, _ = self.lsm.shape
        dest = torch.distributions.Categorical(logits=self.lsm.reshape(-1, E)).sample().view(B_, E)
        return self._bundle(dest, torch.bernoulli(self.p))
    def greedy(self):
        return self._bundle(self.glogits.argmax(-1), (self.p >= 0.5).to(self.lsm.dtype))
    def log_prob(self, a):
        E = self.lsm.shape[-1]
        dest = a[..., :E].argmax(-1)                                        # rows are one-hot
        fire = a[..., E]
        where_lp = self.lsm.gather(2, dest.unsqueeze(-1)).squeeze(-1)
        gate_lp = fire * torch.log(self.p.clamp_min(1e-8)) + (1.0 - fire) * torch.log((1.0 - self.p).clamp_min(1e-8))
        lp = gate_lp + fire * where_lp
        return (lp * self.owned).sum(1)
    def entropy(self):
        p = self.p
        gate_e = -(p * torch.log(p.clamp_min(1e-8)) + (1.0 - p) * torch.log((1.0 - p).clamp_min(1e-8)))
        cat_e = -(self.lsm.exp() * self.lsm.clamp_min(-30.0)).sum(-1)       # bounded (<= ln E)
        ent = gate_e + ENT_DIR_COEF * p * cat_e
        return (ent * self.owned).sum(1)


## 5. World generator (pure Python mirror of official `generate_planets`)

Mirrors `REFERENCE_orbit_wars.py::generate_planets` + `native_worldgen.generate_world`:
4-fold symmetric planet groups (each `[id, owner, x, y, radius, ships, production]`), a
guaranteed `MIN_STATIC_GROUPS=3` static groups via polar sampling, `MIN/MAX_PLANET_GROUPS`
total, then home assignment (`base` group: planet 0 -> player 0 with 10 ships, planet 3 ->
player 1 with 10 ships). `angular_velocity ~ U(0.025, 0.05)`.

**Simplification:** no comet schedule (comets omitted). Worlds are packed straight into the
batched env tensors (no `.owp` files).

In [ ]:
MIN_PLANET_GROUPS = 5
MAX_PLANET_GROUPS = 10
MIN_STATIC_GROUPS = 3
PLANET_CLEARANCE = 7

def _dist(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])

def generate_planets(rng):
    '''Faithful mirror of REFERENCE_orbit_wars.py::generate_planets.
    Returns rows [id, owner, x, y, radius, ships, production].'''
    planets = []
    num_q1 = rng.randint(MIN_PLANET_GROUPS, MAX_PLANET_GROUPS)
    idc = 0
    # Phase 1: guaranteed static groups (polar sampling).
    static_groups = 0
    for _ in range(5000):
        if static_groups >= MIN_STATIC_GROUPS:
            break
        prod = rng.randint(1, 5)
        r = 1 + math.log(prod)
        angle = rng.uniform(0, math.pi / 2)
        min_orbital = ROTATION_RADIUS_LIMIT - r
        max_orbital = (BOARD_SIZE - CENTER - r) / max(math.cos(angle), math.sin(angle))
        if min_orbital > max_orbital:
            continue
        orbital_r = rng.uniform(min_orbital, max_orbital)
        x = CENTER + orbital_r * math.cos(angle)
        y = CENTER + orbital_r * math.sin(angle)
        if x + r > BOARD_SIZE or x - r < 0 or y + r > BOARD_SIZE or y - r < 0:
            continue
        if (BOARD_SIZE - x) - r < 0 or (BOARD_SIZE - y) - r < 0:
            continue
        if (x - CENTER) < r + 5 or (y - CENTER) < r + 5:
            continue
        ships = min(rng.randint(5, 99), rng.randint(5, 99))
        # NOTE: the reference stores rows as [id, owner, y, x, r, ...] (x/y swapped naming),
        # which is just a relabel of the symmetric copies; we keep it identical.
        tps = [
            [idc, -1, y, x, r, ships, prod],
            [idc + 1, -1, BOARD_SIZE - x, y, r, ships, prod],
            [idc + 2, -1, x, BOARD_SIZE - y, r, ships, prod],
            [idc + 3, -1, BOARD_SIZE - y, BOARD_SIZE - x, r, ships, prod],
        ]
        valid = True
        for tp in tps:
            for p in planets:
                if _dist((p[2], p[3]), (tp[2], tp[3])) < p[4] + tp[4] + PLANET_CLEARANCE:
                    valid = False; break
            if not valid:
                break
        if valid:
            planets.extend(tps); idc += 4; static_groups += 1
    # Phase 2: fill remaining groups (normal random loop).
    attempts = 0
    max_attempts = 5000
    has_orbiting = False
    while len(planets) < num_q1 * 4 or (not has_orbiting and attempts < max_attempts):
        attempts += 1
        if attempts >= max_attempts:
            break
        prod = rng.randint(1, 5)
        r = 1 + math.log(prod)
        x = rng.uniform(CENTER + 15, BOARD_SIZE - r - 5)
        y = rng.uniform(CENTER + 15, BOARD_SIZE - r - 5)
        orbital_radius = _dist((x, y), (CENTER, CENTER))
        if orbital_radius < SUN_RADIUS + r + 10:
            continue
        if orbital_radius + r >= ROTATION_RADIUS_LIMIT:
            if x + r > BOARD_SIZE or x - r < 0 or y + r > BOARD_SIZE or y - r < 0:
                continue
        valid = True
        ships = rng.randint(5, 30)
        tps = [
            [idc, -1, y, x, r, ships, prod],
            [idc + 1, -1, BOARD_SIZE - x, y, r, ships, prod],
            [idc + 2, -1, x, BOARD_SIZE - y, r, ships, prod],
            [idc + 3, -1, BOARD_SIZE - y, BOARD_SIZE - x, r, ships, prod],
        ]
        for tp in tps:
            tp_orb = _dist((tp[2], tp[3]), (CENTER, CENTER))
            tp_rot = tp_orb + tp[4] < ROTATION_RADIUS_LIMIT
            for p in planets:
                p_orb = _dist((p[2], p[3]), (CENTER, CENTER))
                p_rot = p_orb + p[4] < ROTATION_RADIUS_LIMIT
                if _dist((p[2], p[3]), (tp[2], tp[3])) < p[4] + tp[4] + PLANET_CLEARANCE:
                    valid = False; break
                if tp_rot != p_rot:
                    if abs(tp_orb - p_orb) < tp[4] + p[4] + PLANET_CLEARANCE:
                        valid = False; break
            if not valid:
                break
        if valid:
            if orbital_radius + r < ROTATION_RADIUS_LIMIT:
                has_orbiting = True
            planets.extend(tps); idc += 4
    return planets

def generate_world(seed):
    '''Mirror of native_worldgen.generate_world (comets dropped).'''
    rng = random.Random(seed)
    angular_velocity = rng.uniform(0.025, 0.05)
    planets = generate_planets(rng)
    num_groups = len(planets) // 4
    base = -1
    if num_groups > 0:
        base = rng.randint(0, num_groups - 1) * 4
        planets[base][1] = 0;      planets[base][5] = 10       # player 0 home
        planets[base + 3][1] = 1;  planets[base + 3][5] = 10   # player 1 home
    # v8: remember the home group; env.reset(n_players=4) re-seats owners 0..3 on base..base+3
    # exactly like the official engine's 4-player branch (same group => fair under 4-fold symmetry).
    w = {"planets": planets, "angular_velocity": angular_velocity, "home_base": base}
    if COMET_OFFICIAL:
        attach_official_comets(w, seed)
    return w

# ---- OFFICIAL comet paths (numpy-vectorized port of REFERENCE_orbit_wars.generate_comet_paths;
# ---- same math/draw-order/reject-semantics, bit-exact-verified by scripts/test_official_comets.py)
def _comet_paths_official(initial_planets, angular_velocity, spawn_step, comet_speed, rng):
    """Returns 4 symmetric waypoint paths (list of [x,y], one waypoint per tick) or None."""
    stat, orb = [], []
    for p in initial_planets:
        pr = math.sqrt((p[2] - CENTER) ** 2 + (p[3] - CENTER) ** 2)
        (orb if pr + p[4] < ROTATION_RADIUS_LIMIT else stat).append(p)
    stat_xy = np.array([[p[2], p[3]] for p in stat], np.float64).reshape(-1, 2)
    stat_rad = np.array([p[4] for p in stat], np.float64)
    orb_r = np.array([math.sqrt((p[2] - CENTER) ** 2 + (p[3] - CENTER) ** 2) for p in orb], np.float64)
    orb_a0 = np.array([math.atan2(p[3] - CENTER, p[2] - CENTER) for p in orb], np.float64)
    orb_rad = np.array([p[4] for p in orb], np.float64)
    num = 5000
    t_arr = 0.3 * math.pi + 1.4 * math.pi * np.arange(num) / (num - 1)
    cos_t, sin_t = np.cos(t_arr), np.sin(t_arr)
    for _ in range(300):
        e = rng.uniform(0.75, 0.93)
        a = rng.uniform(60, 150)
        if a * (1 - e) < SUN_RADIUS + COMET_RADIUS:
            continue
        b = a * math.sqrt(1 - e ** 2)
        c_val = a * e
        phi = rng.uniform(math.pi / 6, math.pi / 3)
        ex = c_val + a * cos_t; ey = b * sin_t
        cp, sp = math.cos(phi), math.sin(phi)
        x = CENTER + ex * cp - ey * sp
        y = CENTER + ex * sp + ey * cp
        dx = x[1:] - x[:-1]; dy = y[1:] - y[:-1]
        cum = np.cumsum(np.sqrt(dx * dx + dy * dy))            # cum[i-1] = arc length at dense[i]
        nk = int(cum[-1] // comet_speed) + 1
        sel = np.searchsorted(cum, comet_speed * np.arange(1, nk + 1), side="left") + 1
        sel = sel[sel < num]
        px = np.concatenate(([x[0]], x[sel])); py = np.concatenate(([y[0]], y[sel]))
        on = (px >= 0) & (px <= BOARD_SIZE) & (py >= 0) & (py <= BOARD_SIZE)
        if not on.any():
            continue
        i0 = int(np.argmax(on)); i1 = int(len(on) - 1 - np.argmax(on[::-1]))
        vx = px[i0:i1 + 1]; vy = py[i0:i1 + 1]                  # contiguous on-board SPAN (incl. interior)
        K = len(vx)
        if not (5 <= K <= 40):
            continue
        if (np.sqrt((vx - CENTER) ** 2 + (vy - CENTER) ** 2) < SUN_RADIUS + COMET_RADIUS).any():
            continue
        sym_x = np.stack([vy, BOARD_SIZE - vx, vx, BOARD_SIZE - vy])          # (4,K)
        sym_y = np.stack([vx, vy, BOARD_SIZE - vy, BOARD_SIZE - vx])
        if len(stat_xy):
            d = np.sqrt((sym_x[:, :, None] - stat_xy[None, None, :, 0]) ** 2
                        + (sym_y[:, :, None] - stat_xy[None, None, :, 1]) ** 2)
            if (d < stat_rad[None, None, :] + (COMET_RADIUS + 0.5)).any():
                continue
        if len(orb_r):
            gs = spawn_step - 1 + np.arange(K)                                # (K,)
            ang = orb_a0[:, None] + angular_velocity * gs[None, :]            # (P,K)
            ox = CENTER + orb_r[:, None] * np.cos(ang)
            oy = CENTER + orb_r[:, None] * np.sin(ang)
            d = np.sqrt((sym_x[:, :, None] - ox.T[None, :, :]) ** 2
                        + (sym_y[:, :, None] - oy.T[None, :, :]) ** 2)        # (4,K,P)
            if (d < orb_rad[None, None, :] + COMET_RADIUS).any():
                continue
        return [
            [[float(vy[k]), float(vx[k])] for k in range(K)],
            [[float(BOARD_SIZE - vx[k]), float(vy[k])] for k in range(K)],
            [[float(vx[k]), float(BOARD_SIZE - vy[k])] for k in range(K)],
            [[float(BOARD_SIZE - vy[k]), float(BOARD_SIZE - vx[k])] for k in range(K)],
        ]
    return None


def attach_official_comets(world, seed):
    """Precompute the official comet schedule for one world (CPU). Stores padded arrays:
    comet_paths (NS,4,COMET_MAX_LEN,2) f32 / comet_len (NS,) / comet_ships (NS,) where
    NS = len(COMET_SPAWN_STEPS). Draw order matches the engine exactly (paths, then ships)."""
    NS = len(COMET_SPAWN_STEPS)
    cp = np.zeros((NS, 4, COMET_MAX_LEN, 2), np.float32)
    cl = np.zeros((NS,), np.int64)
    cs = np.zeros((NS,), np.float32)
    for e, s in enumerate(COMET_SPAWN_STEPS):
        rng = random.Random(f"orbit_wars-comet-{seed}-{s}")
        paths = _comet_paths_official(world["planets"], world["angular_velocity"], s, COMET_SPEED, rng)
        if not paths:
            continue
        ships = min(rng.randint(1, 99), rng.randint(1, 99), rng.randint(1, 99), rng.randint(1, 99))
        L = len(paths[0])
        for m in range(4):
            for k, (x, y) in enumerate(paths[m][:COMET_MAX_LEN]):
                cp[e, m, k, 0] = x; cp[e, m, k, 1] = y
        cl[e] = min(L, COMET_MAX_LEN)
        cs[e] = float(ships)
    world["comet_paths"] = cp; world["comet_len"] = cl; world["comet_ships"] = cs
    return world


def make_world_pool(n, base_seed=0):
    return [generate_world(base_seed + i) for i in range(n)]

# quick sanity: one world's shape
_w = generate_world(0)
print("world 0:", len(_w["planets"]), "planets | ang_vel = %.4f" % _w["angular_velocity"])

## 6. Batched env (mirrors `rl/gpu_env.cpp`)

Structure-of-Arrays over `B` envs, leading dim `B`. Integer quantities (ships, production,
owner, step) live in float32 (exact below `2^24`, cheaper on GPU). Planets in fixed slots
`[0, PLANET_CAP)`; fleets in a fixed pool `[0, FLEET_CAP)` with an alive mask. **No comet
slots** (comets omitted).

`reset` packs a list of world dicts into the device tensors and zeroes runtime state.

**v8**: `reset(worlds, n_players)` -- for 4p FFA, owners 0..3 are seated on the official symmetric home group (`home_base`), 10 ships each, exactly like the engine's 4-player branch.


In [ ]:
class GpuEnv:
    def __init__(self, planet_cap, fleet_cap, episode_steps, ship_speed, device):
        self.Ec = planet_cap
        self.Fc = fleet_cap
        self.T = episode_steps
        self.vmax = ship_speed
        self.dev = device
        self.B = 0
        self.n_players = 2

    def reset(self, worlds, n_players=2):
        self.n_players = int(n_players)
        B = len(worlds)
        self.B = B
        Ec, Fc = self.Ec, self.Fc
        dev = self.dev
        z = lambda *s: torch.zeros(s, dtype=DTYPE, device=dev)
        # planets (B, Ec)
        self.p_alive = z(B, Ec)
        self.p_owner = torch.full((B, Ec), -1.0, dtype=DTYPE, device=dev)
        self.p_x = z(B, Ec); self.p_y = z(B, Ec); self.p_radius = z(B, Ec)
        self.p_ships = z(B, Ec); self.p_prod = z(B, Ec); self.p_is_comet = z(B, Ec)
        self.p_init_x = z(B, Ec); self.p_init_y = z(B, Ec); self.p_rotates = z(B, Ec)
        self.p_comet_vx = z(B, Ec); self.p_comet_vy = z(B, Ec)   # comet straight-line velocity
        # fill from worlds (CPU numpy then upload once)
        pa = np.zeros((B, Ec), np.float32); po = np.full((B, Ec), -1.0, np.float32)
        px = np.zeros((B, Ec), np.float32); py = np.zeros((B, Ec), np.float32)
        pr = np.zeros((B, Ec), np.float32); ps = np.zeros((B, Ec), np.float32)
        pp = np.zeros((B, Ec), np.float32); pix = np.zeros((B, Ec), np.float32)
        piy = np.zeros((B, Ec), np.float32); prot = np.zeros((B, Ec), np.float32)
        angv = np.zeros((B,), np.float32)
        for b, w in enumerate(worlds):
            pls = w["planets"][:Ec]
            for i, pl in enumerate(pls):
                pid, owner, x, y, rad, sh, prod = pl
                pa[b, i] = 1.0; po[b, i] = float(owner)
                px[b, i] = x; py[b, i] = y; pr[b, i] = rad
                ps[b, i] = sh; pp[b, i] = prod
                pix[b, i] = x; piy[b, i] = y
                rr = math.hypot(x - CENTER, y - CENTER)
                prot[b, i] = 1.0 if (rr + rad < ROTATION_RADIUS_LIMIT) else 0.0
            angv[b] = w["angular_velocity"]
            if self.n_players >= 4:               # official 4p seating: owners 0..3, 10 ships each
                hb = int(w.get("home_base", -1))
                if 0 <= hb and hb + 3 < Ec:
                    for j in range(4):
                        po[b, hb + j] = float(j); ps[b, hb + j] = 10.0
        t = lambda a: torch.from_numpy(a).to(dev)
        self.p_alive = t(pa); self.p_owner = t(po); self.p_x = t(px); self.p_y = t(py)
        self.p_radius = t(pr); self.p_ships = t(ps); self.p_prod = t(pp)
        self.p_init_x = t(pix); self.p_init_y = t(piy); self.p_rotates = t(prot)
        self.p_is_comet = z(B, Ec)
        # fleets (B, Fc)
        self.f_alive = z(B, Fc); self.f_owner = z(B, Fc); self.f_x = z(B, Fc)
        self.f_y = z(B, Fc); self.f_angle = z(B, Fc); self.f_ships = z(B, Fc)
        self.f_seq = z(B, Fc)
        # official comet schedule (padded waypoint playback)
        NS = len(COMET_SPAWN_STEPS)
        if COMET_OFFICIAL and worlds and ('comet_paths' in worlds[0]):
            cp = np.stack([w['comet_paths'] for w in worlds])           # (B,NS,4,L,2)
            cl = np.stack([w['comet_len'] for w in worlds])             # (B,NS)
            cs = np.stack([w['comet_ships'] for w in worlds])           # (B,NS)
            self.c_paths = torch.from_numpy(cp).to(dev)
            self.c_len = torch.from_numpy(cl).to(dev)
            self.c_ships = torch.from_numpy(cs).to(dev)
            self.c_slot = torch.full((B, NS, 4), -1, dtype=torch.long, device=dev)
        else:
            self.c_paths = None
        # per-env
        self.ang_vel = t(angv)
        self.step_ct = torch.zeros(B, dtype=DTYPE, device=dev)
        self.done = torch.zeros(B, dtype=DTYPE, device=dev)

## 7. Env -- `fleet_target_batch` + `encode`

`fleet_target_batch`: closed-form ray-vs-disk + sun occlusion for `M` virtual fleets/env vs
the current planets (mirrors `gpu_env.cpp::fleet_target_batch`). Used by `encode` (threat
features, `M=Fc`) and by the valid-launch check.

`encode(ego)`: the batched mirror of `encode_obs`. Planets stay in fixed slots (the per-planet
actor is permutation-equivariant). Returns entities `(B,Ec,F)`, entity_mask, action_mask,
globals `(B,Ec...)`.

In [ ]:
def fleet_target_batch(env, fx, fy, fang, fships, vmax):
    '''fx,fy,fang,fships: (B,M). Returns tgt (B,M) long (planet slot or -1), eta (B,M).'''
    dx = torch.cos(fang).unsqueeze(2)           # (B,M,1)
    dy = torch.sin(fang).unsqueeze(2)
    pxr = env.p_x.unsqueeze(1)                  # (B,1,Ec)
    pyr = env.p_y.unsqueeze(1)
    rr = (env.p_radius * env.p_radius).unsqueeze(1)
    alive = (env.p_alive > 0.5).unsqueeze(1)
    ox = fx.unsqueeze(2) - pxr                  # (B,M,Ec)
    oy = fy.unsqueeze(2) - pyr
    tca = -(ox * dx + oy * dy)
    perp2 = ox * ox + oy * oy - tca * tca
    hit = (tca >= 0.0) & (perp2 <= rr) & alive
    t_int = (tca - torch.sqrt((rr - perp2).clamp_min(0.0))).clamp_min(0.0)
    tvals = torch.where(hit, t_int, torch.full_like(t_int, BIG))
    best_t, best_e = tvals.min(2)               # (B,M)
    any_hit = best_t < BIG
    # sun occlusion
    sx = fx - CENTER; sy = fy - CENTER
    dxx = torch.cos(fang); dyy = torch.sin(fang)
    tcs = -(sx * dxx + sy * dyy)
    sperp2 = sx * sx + sy * sy - tcs * tcs
    sr = SUN_RADIUS * SUN_RADIUS
    t_sun = tcs - torch.sqrt((sr - sperp2).clamp_min(0.0))
    sun_block = (tcs >= 0.0) & (sperp2 <= sr) & (t_sun >= 0.0) & (t_sun < best_t)
    valid = any_hit & (~sun_block)
    tgt = torch.where(valid, best_e, torch.full_like(best_e, -1))
    eta = best_t / fleet_speed_t(fships, vmax)
    eta = torch.where(valid, eta, torch.zeros_like(eta))
    return tgt, eta


def env_encode(env, ego=0):
    """v8: N-player aware. EVERY non-ego owner collapses to the single 'enemy' code and the
    enemy aggregates pool ALL opponents -- identical encodings in 2p, and exactly the deployed
    Kaggle 4p adapter view in FFA (the policy plays everyone-vs-me). v7 weights stay drop-in
    compatible (F_DIM 101); the per-seat one-hot redesign is v9."""
    B, Ec, Fc = env.B, env.Ec, env.Fc
    dev = env.dev
    alive = env.p_alive > 0.5
    av = env.p_alive
    owner = env.p_owner
    dxc = env.p_x - CENTER; dyc = env.p_y - CENTER
    dist = torch.sqrt(dxc * dxc + dyc * dyc)
    comet = env.p_is_comet > 0.5
    rotating = (~comet) & ((dist + env.p_radius) < ROTATION_RADIUS_LIMIT)
    angv = env.ang_vel.unsqueeze(1)
    vmag = torch.where(rotating, angv.abs() * dist, torch.zeros_like(dist))
    cw = torch.where(rotating,
                     torch.where(angv >= 0.0, torch.ones_like(dist), -torch.ones_like(dist)),
                     torch.zeros_like(dist))
    # ownership code: 0 neutral, 1 ego, 2 ANY other player (2p-identical; 4p collapses)
    own_code = torch.where(owner < 0.0, torch.zeros_like(owner),
                           torch.where(owner == float(ego), torch.ones_like(owner),
                                       torch.full_like(owner, 2.0)))
    b0 = env.p_x / BOARD_SIZE
    b1 = env.p_y / BOARD_SIZE
    b2 = vmag / THREAT_MAX_SPEED
    b3 = cw
    b4 = env.p_radius / 3.0
    b5 = env.p_prod / 5.0
    b6 = ship_log_t(env.p_ships)
    b7 = own_code
    b8 = dist / DIAG_HALF
    b9 = comet.to(DTYPE)
    actable = (owner == float(ego)) & alive & (env.p_ships > 0.0)
    b10 = actable.to(DTYPE)
    body = torch.stack([b0, b1, b2, b3, b4, b5, b6, b7, b8, b9, b10], 2)  # (B,Ec,11)

    # threat: per planet, N_SOON soonest + N_BIG largest inbound fleets
    ftgt, feta = fleet_target_batch(env, env.f_x, env.f_y, env.f_angle, env.f_ships, env.vmax)
    falive = env.f_alive > 0.5
    fsign = torch.where(env.f_owner == float(ego), torch.ones_like(env.f_ships),
                        -torch.ones_like(env.f_ships))
    fslog = ship_log_t(env.f_ships.abs())
    slot = torch.arange(Ec, dtype=DTYPE, device=dev).view(1, Ec, 1)
    tgtf = ftgt.to(DTYPE).unsqueeze(1)                       # (B,1,Fc)
    targeting = (tgtf == slot) & (ftgt >= 0).unsqueeze(1) & falive.unsqueeze(1)  # (B,Ec,Fc)
    W = float(1 << 20)
    eta_mat = torch.where(targeting, feta.unsqueeze(1), torch.full((1,), BIG, device=dev))
    absf = env.f_ships.abs().unsqueeze(1)
    seqf = env.f_seq.unsqueeze(1)
    key_big = torch.where(targeting, absf * W - seqf, torch.full((1,), -BIG, device=dev))
    eta_top_v, eta_top_i = torch.topk(eta_mat, N_SOON, 2, largest=False)
    big_top_v, big_top_i = torch.topk(key_big, N_BIG, 2, largest=True)
    idx = torch.cat([eta_top_i, big_top_i], 2)              # (B,Ec,7)
    valid = torch.cat([eta_top_v < BIG * 0.5, big_top_v > -BIG * 0.5], 2)
    def pick(src):
        return src.unsqueeze(1).expand(B, Ec, Fc).gather(2, idx)
    zt = torch.zeros(B, Ec, N_THREAT_FLEETS, device=dev)
    sgn = torch.where(valid, pick(fsign), zt)
    etf = torch.where(valid, pick(feta) / THREAT_ETA_SCALE, zt)
    slg = torch.where(valid, pick(fslog), zt)
    threat = torch.stack([sgn, etf, slg], 3).reshape(B, Ec, 3 * N_THREAT_FLEETS)  # (B,Ec,21)

    entities = torch.cat([body, threat], 2) * av.unsqueeze(2)   # zero dead slots
    entity_mask = av
    action_mask = b10

    # globals (B,10)
    def psum(mask):
        return (env.p_ships * mask.to(DTYPE)).sum(1)
    mine = (owner == float(ego)) & alive
    en = (owner >= 0.0) & (owner != float(ego)) & alive    # ALL opponents pooled
    neu = (owner < 0.0) & alive
    my_ships = psum(mine); en_ships = psum(en)
    my_pl = mine.to(DTYPE).sum(1); en_pl = en.to(DTYPE).sum(1); neu_pl = neu.to(DTYPE).sum(1)
    npl = av.sum(1); total = npl.clamp_min(1.0)
    my_fleet = (env.f_ships * (env.f_owner == float(ego)).to(DTYPE) * falive.to(DTYPE)).sum(1)
    en_fleet = (env.f_ships * (env.f_owner != float(ego)).to(DTYPE) * falive.to(DTYPE)).sum(1)
    ME = 40.0
    g0 = env.step_ct / max(1, env.T)
    g1 = env.ang_vel * 10.0
    g2 = ship_log_t(my_ships); g3 = ship_log_t(en_ships)
    g4 = my_pl / total; g5 = en_pl / total; g6 = neu_pl / total
    g7 = ship_log_t(my_fleet); g8 = ship_log_t(en_fleet)
    g9 = torch.minimum(npl, torch.full_like(npl, ME)) / ME
    globals_ = torch.stack([g0, g1, g2, g3, g4, g5, g6, g7, g8, g9], 1)
    return entities, entity_mask, action_mask, globals_

## 8. Env -- `launch_fleets` + scripted opponents

`launch_fleets`: scatter committed launches into free fleet slots (mirrors
`gpu_env.cpp::launch_fleets`; ship deduction happens in `step`). `opponent_action`: noop /
random / starter (mirrors `gpu_env.cpp::opponent_action`).

In [ ]:
def launch_fleets(env, owner, from_slot, angle, ships, commit, seq):
    '''Append committed launches (all (B,L)) into the fleet pool, distinct free slots.'''
    B, Fc = env.B, env.Fc
    dev = env.dev
    L = from_slot.shape[1]
    free = env.f_alive < 0.5
    freef = free.to(DTYPE)
    fr = torch.cumsum(freef, 1) - 1.0
    n_free = freef.sum(1)
    idx = torch.where(free, fr.long(), torch.full_like(fr.long(), Fc))
    rank_to_slot = torch.full((B, Fc + 1), Fc, dtype=torch.long, device=dev)
    slot_src = torch.arange(Fc, dtype=torch.long, device=dev).unsqueeze(0).expand(B, Fc)
    rank_to_slot.scatter_(1, idx, slot_src)
    rank_to_slot = rank_to_slot[:, :Fc]
    commitb = commit > 0.5
    crank = (torch.cumsum(commit.to(DTYPE), 1) - 1.0).long()
    place = commitb & (crank < n_free.unsqueeze(1).long())
    gslot = rank_to_slot.gather(1, crank.clamp(0, Fc - 1))
    dump = torch.full_like(gslot, Fc)
    wslot = torch.where(place, gslot, dump)
    def scatter_into(field, vals):
        aug = torch.cat([field, torch.zeros(B, 1, dtype=DTYPE, device=dev)], 1)
        aug.scatter_(1, wslot, vals)
        return aug[:, :Fc]
    fs = from_slot.clamp(0, env.p_x.shape[1] - 1)
    opx = env.p_x.gather(1, fs); opy = env.p_y.gather(1, fs); orad = env.p_radius.gather(1, fs)
    sx = opx + torch.cos(angle) * (orad + 0.1)
    sy = opy + torch.sin(angle) * (orad + 0.1)
    onesL = torch.ones(B, L, dtype=DTYPE, device=dev)
    env.f_alive = scatter_into(env.f_alive, onesL)
    env.f_owner = scatter_into(env.f_owner, owner)
    env.f_x = scatter_into(env.f_x, sx)
    env.f_y = scatter_into(env.f_y, sy)
    env.f_angle = scatter_into(env.f_angle, angle)
    env.f_ships = scatter_into(env.f_ships, ships)
    env.f_seq = scatter_into(env.f_seq, seq)


def opponent_action(env, opponent, pid=1):
    '''Scripted launches for seat `pid` (v8: any seat in a 2p/4p game): one per owned planet, half garrison (>=20).
    0=random heading, 1=starter (nearest static non-owned), 2=noop, 3=medium (starter++: nearest incl. rotating, lead-aim),
    4=greedy (nearest beatable non-owned planet within CAPTURE_RADIUS, just-enough ships),
    5=intermediate (in-flight-aware capture from nearest capable source + counter + rebalance + comet-escape).
    -> angle,ships,commit (B,Ec).'''
    B, Ec = env.B, env.Ec
    dev = env.dev
    min_ships = 20.0
    angle = torch.zeros(B, Ec, dtype=DTYPE, device=dev)
    ships = torch.zeros(B, Ec, dtype=DTYPE, device=dev)
    commit = torch.zeros(B, Ec, dtype=DTYPE, device=dev)
    if opponent == 2:
        return angle, ships, commit
    half = torch.floor(env.p_ships / 2.0)
    base = (env.p_owner == float(pid)) & (env.p_alive > 0.5) & (half >= min_ships)
    if opponent == 0:  # random heading
        angle = torch.rand(B, Ec, dtype=DTYPE, device=dev) * (2.0 * PI)
        ships = torch.where(base, half, torch.zeros_like(half))
        commit = base.to(DTYPE)
        return angle, ships, commit
    if opponent == 3:  # medium (starter++): nearest non-owned planet INCL. rotating, lead-aimed at its intercept
        sx = env.p_x.unsqueeze(2); sy = env.p_y.unsqueeze(2)             # (B,Ec,1) source
        tx = env.p_x.unsqueeze(1); ty = env.p_y.unsqueeze(1)            # (B,1,Ec) dest
        vt = (env.p_alive > 0.5) & (env.p_owner != float(pid))                 # (B,Ec) valid targets (rotating allowed)
        d = torch.sqrt((sx - tx) ** 2 + (sy - ty) ** 2)
        dmask = torch.where(vt.unsqueeze(1), d, torch.full_like(d, BIG))
        bestd, tgt = dmask.min(2)                                       # (B,Ec) nearest valid target per source
        has_tgt = bestd < BIG * 0.5
        dpx = env.p_x.gather(1, tgt); dpy = env.p_y.gather(1, tgt)      # (B,Ec) target position
        if LEAD_TARGET:                                                 # lead the (possibly orbiting) target
            rdx = dpx - CENTER; rdy = dpy - CENTER
            r_d = torch.sqrt(rdx * rdx + rdy * rdy)
            phi0 = torch.atan2(rdy, rdx)
            drot = env.p_rotates.gather(1, tgt) > 0.5
            w = torch.where(drot, env.ang_vel.view(B, 1), torch.zeros(B, 1, device=dev, dtype=DTYPE))
            off = env.p_radius + 0.1
            v = fleet_speed_t(half.clamp_min(1.0), env.vmax)
            t = (torch.sqrt((dpx - env.p_x) ** 2 + (dpy - env.p_y) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
            for _ in range(16):
                phi = phi0 + w * t
                ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
                t = (torch.sqrt((ix - env.p_x) ** 2 + (iy - env.p_y) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
            phi = phi0 + w * t
            ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
            angle = torch.atan2(iy - env.p_y, ix - env.p_x)
        else:
            angle = torch.atan2(dpy - env.p_y, dpx - env.p_x)
        go = base & has_tgt
        ships = torch.where(go, half, torch.zeros_like(half))
        commit = go.to(DTYPE)
        return angle, ships, commit
    if opponent == 4:  # greedy: capture the NEAREST non-owned planet within CAPTURE_RADIUS we can already beat
        sx = env.p_x.unsqueeze(2); sy = env.p_y.unsqueeze(2)            # (B,Ec,1) source
        tx = env.p_x.unsqueeze(1); ty = env.p_y.unsqueeze(1)           # (B,1,Ec) dest
        d = torch.sqrt((sx - tx) ** 2 + (sy - ty) ** 2)                # (B,Ec,Ec)
        vt = (env.p_alive > 0.5) & (env.p_owner != float(pid))                # (B,Ec) non-owned alive
        can_beat = env.p_ships.unsqueeze(2) > env.p_ships.unsqueeze(1) # (B,src,dst) src ships > dst ships
        cand = vt.unsqueeze(1) & can_beat & (d <= CAPTURE_RADIUS)      # capturable & in range
        dmask = torch.where(cand, d, torch.full_like(d, BIG))
        bestd, tgt = dmask.min(2)                                       # (B,Ec) nearest capturable target
        has_tgt = bestd < BIG * 0.5
        tgt_ships = env.p_ships.gather(1, tgt)                          # (B,Ec)
        n = torch.minimum(env.p_ships, torch.floor(tgt_ships) + CAPTURE_MARGIN)  # just enough to capture (+ buffer)
        dpx = env.p_x.gather(1, tgt); dpy = env.p_y.gather(1, tgt)      # (B,Ec) target position
        if LEAD_TARGET:                                                 # lead the (possibly orbiting) target
            rdx = dpx - CENTER; rdy = dpy - CENTER
            r_d = torch.sqrt(rdx * rdx + rdy * rdy)
            phi0 = torch.atan2(rdy, rdx)
            drot = env.p_rotates.gather(1, tgt) > 0.5
            w = torch.where(drot, env.ang_vel.view(B, 1), torch.zeros(B, 1, device=dev, dtype=DTYPE))
            off = env.p_radius + 0.1
            v = fleet_speed_t(n.clamp_min(1.0), env.vmax)
            t = (torch.sqrt((dpx - env.p_x) ** 2 + (dpy - env.p_y) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
            for _ in range(16):
                phi = phi0 + w * t
                ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
                t = (torch.sqrt((ix - env.p_x) ** 2 + (iy - env.p_y) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
            phi = phi0 + w * t
            ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
            angle = torch.atan2(iy - env.p_y, ix - env.p_x)
        else:
            angle = torch.atan2(dpy - env.p_y, dpx - env.p_x)
        own_ok = (env.p_owner == float(pid)) & (env.p_alive > 0.5) & (env.p_ships > 0.0)
        go = own_ok & has_tgt & (n >= 1.0)
        ships = torch.where(go, n, torch.zeros_like(n))
        commit = go.to(DTYPE)
        return angle, ships, commit
    if opponent == 5:  # intermediate: in-flight-aware capture from nearest capable source (cascade) + rebalance
        owned = (env.p_owner == float(pid)) & (env.p_alive > 0.5) & (env.p_ships > 0.0)   # (B,Ec) sources that can act
        sx = env.p_x.unsqueeze(2); sy = env.p_y.unsqueeze(2)
        tx = env.p_x.unsqueeze(1); ty = env.p_y.unsqueeze(1)
        d = torch.sqrt((sx - tx) ** 2 + (sy - ty) ** 2)                # (B,src,dst)
        srcrange = torch.arange(Ec, device=dev).view(1, Ec, 1)
        # capture (in-flight aware): per target, needed = garrison + enemy_inbound - friendly_inbound + margin
        ftgt, _feta = fleet_target_batch(env, env.f_x, env.f_y, env.f_angle, env.f_ships, env.vmax)
        f_ok = (env.f_alive > 0.5) & (ftgt >= 0)
        tslot_f = ftgt.clamp(0, Ec - 1)
        in_fr = torch.zeros(B, Ec, device=dev, dtype=DTYPE)            # friendly ships already inbound per planet
        in_en = torch.zeros(B, Ec, device=dev, dtype=DTYPE)            # enemy ships inbound per planet
        in_fr.scatter_add_(1, tslot_f, env.f_ships * ((env.f_owner == float(pid)) & f_ok).to(DTYPE))
        in_en.scatter_add_(1, tslot_f, env.f_ships * ((env.f_owner != float(pid)) & f_ok).to(DTYPE))
        vt = (env.p_alive > 0.5) & (env.p_owner != float(pid))
        needed_t = torch.floor(env.p_ships) + torch.ceil(in_en) - torch.floor(in_fr) + CAPTURE_MARGIN   # (B,Ec)
        open_t = vt & (needed_t > 0.0)                                 # not already covered by friendly inbound
        can_supply = env.p_ships.unsqueeze(2) >= needed_t.unsqueeze(1) # (B,src,dst) source can cover the requirement
        capable = owned.unsqueeze(2) & open_t.unsqueeze(1) & can_supply
        d_cap = torch.where(capable, d, torch.full_like(d, BIG))
        nearest_cap = d_cap.argmin(1)                                  # (B,dst) nearest CAPABLE source (cascade)
        target_has = d_cap.min(1).values < BIG * 0.5
        assigned = (nearest_cap.unsqueeze(1) == srcrange) & target_has.unsqueeze(1)
        d_asg = torch.where(assigned, d, torch.full_like(d, BIG))
        cap_d, cap_tgt = d_asg.min(2)                                  # (B,Ec) source's nearest assigned target
        cap_has = cap_d < BIG * 0.5
        cap_n = torch.minimum(env.p_ships, needed_t.gather(1, cap_tgt))   # send exactly the requirement
        # rebalance: feed the poorest owned planet WITHIN REBALANCE_RADIUS (local) when surplus exceeds a percentage
        eyeE = (torch.eye(Ec, device=dev, dtype=DTYPE).unsqueeze(0) > 0.5)
        reb_cand = owned.unsqueeze(1) & (d <= REBALANCE_RADIUS) & (~eyeE)   # (B,src,dst) owned, in-radius, not self
        reb_dst_ships = torch.where(reb_cand, env.p_ships.unsqueeze(1), torch.full_like(d, BIG))
        poor_ships, poor_idx = reb_dst_ships.min(2)                    # (B,Ec) poorest owned-in-radius per source
        has_poor = poor_ships < BIG * 0.5
        gap = env.p_ships - poor_ships
        reb_ok = owned & (~cap_has) & has_poor & (gap > REBALANCE_PCT * env.p_ships.clamp_min(1.0))
        reb_n = torch.floor(gap * 0.5)
        reb_tgt = poor_idx
        # comet escape (top priority): owned comets evacuate ALL ships to the nearest owned non-comet planet
        safe_dst = (owned & (env.p_is_comet < 0.5)).unsqueeze(1) & (~eyeE)   # (B,src,dst) dest owned non-comet, not self
        d_evac = torch.where(safe_dst, d, torch.full_like(d, BIG))
        evac_d, evac_tgt = d_evac.min(2)
        evac_ok = owned & (env.p_is_comet > 0.5) & (evac_d < BIG * 0.5) & (env.p_ships >= 1.0)
        # priority: comet-escape > capture > rebalance
        use_cap = (~evac_ok) & cap_has
        use_reb = (~evac_ok) & (~cap_has) & reb_ok
        tgt = torch.where(evac_ok, evac_tgt, torch.where(use_cap, cap_tgt, reb_tgt))
        nn = torch.where(evac_ok, env.p_ships, torch.where(use_cap, cap_n, torch.where(use_reb, reb_n, torch.zeros_like(reb_n))))
        go = evac_ok | use_cap | use_reb
        dpx = env.p_x.gather(1, tgt); dpy = env.p_y.gather(1, tgt)
        if LEAD_TARGET:
            rdx = dpx - CENTER; rdy = dpy - CENTER
            r_d = torch.sqrt(rdx * rdx + rdy * rdy)
            phi0 = torch.atan2(rdy, rdx)
            drot = env.p_rotates.gather(1, tgt) > 0.5
            w = torch.where(drot, env.ang_vel.view(B, 1), torch.zeros(B, 1, device=dev, dtype=DTYPE))
            off = env.p_radius + 0.1
            v = fleet_speed_t(nn.clamp_min(1.0), env.vmax)
            t = (torch.sqrt((dpx - env.p_x) ** 2 + (dpy - env.p_y) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
            for _ in range(16):
                phi = phi0 + w * t
                ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
                t = (torch.sqrt((ix - env.p_x) ** 2 + (iy - env.p_y) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
            phi = phi0 + w * t
            ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
            angle = torch.atan2(iy - env.p_y, ix - env.p_x)
        else:
            angle = torch.atan2(dpy - env.p_y, dpx - env.p_x)
        ships = torch.where(go, nn, torch.zeros_like(nn))
        commit = go.to(DTYPE)
        return angle, ships, commit
    # opponent == 1: starter -- fire at nearest static non-owned planet
    distc = torch.sqrt((env.p_x - CENTER) ** 2 + (env.p_y - CENTER) ** 2)
    is_static = (distc + env.p_radius) >= ROTATION_RADIUS_LIMIT
    valid_tgt = is_static & (env.p_alive > 0.5) & (env.p_owner != float(pid))
    sx = env.p_x.unsqueeze(2); sy = env.p_y.unsqueeze(2)
    tx = env.p_x.unsqueeze(1); ty = env.p_y.unsqueeze(1)
    d = torch.sqrt((sx - tx) ** 2 + (sy - ty) ** 2)
    dmask = torch.where(valid_tgt.unsqueeze(1), d, torch.full_like(d, BIG))
    bestd, bestidx = dmask.min(2)
    has_tgt = bestd < BIG * 0.5
    btx = env.p_x.gather(1, bestidx); bty = env.p_y.gather(1, bestidx)
    angle = torch.atan2(bty - env.p_y, btx - env.p_x)
    go = base & has_tgt
    ships = torch.where(go, half, torch.zeros_like(half))
    commit = go.to(DTYPE)
    return angle, ships, commit

## 9. Env -- `step` (one tick; v5 target decode = atan2 toward the destination planet)
Action is (dest_slot, phi) per planet; heading aims straight at the destination
(`atan2(p_dest - p_src)`). One launch per planet. Self-play opponent decodes identically.

**v8**: `env_step(..., seats=[...])` takes any mix of scripted/neural seats for pids 1..N-1; combat is the official N-player top-vs-second resolve. v7 2p calls (`env_step(env, a, opp, opp_action)`) still work and are bit-exact.


In [ ]:
class StepOut: pass


def _decode_target(env, action, legal):
    '''Gated-allocation decode. action (B,Ec,Ec+1): [...,:Ec]=WHERE allocation rows, [...,Ec]=fire{0,1}.
    A planet FIRES (full garrison, routed by its renormalised off-diagonal WHERE row) or HOLDS. ships[s,d]
    = floor(Ahat[s,d]*S_s) for d!=s, to ALIVE dests, >= MIN_LAUNCH_SHIPS, only if fire[s]=1; the rest stay
    home. Each launch aims at the orbiting intercept. Returns angle/ships/can (B,Ec,Ec), valid/invalid/launches (B,).'''
    B, Ec, dev = env.B, env.Ec, env.dev
    A = action[..., :Ec]                                        # (B,Ec,Ec) WHERE allocation
    fire = action[..., Ec]                                      # (B,Ec) launch gate in {0,1}
    S = env.p_ships                                             # (B,Ec)
    eye = torch.eye(Ec, dtype=DTYPE, device=dev).unsqueeze(0)   # (1,Ec,Ec)
    A_send = A * (1.0 - eye)                                    # drop self-diagonal (s->s meaningless)
    A_send = A_send / A_send.sum(2, keepdim=True).clamp_min(1e-6)   # renormalise -> fire = FULL garrison out
    raw = A_send * (S * fire).unsqueeze(2)                      # (B,Ec,Ec) ships src->dst, zero if held
    dest_alive = (env.p_alive > 0.5).to(DTYPE).unsqueeze(1)     # (B,1,Ec)
    gate = legal.to(DTYPE).unsqueeze(2) * dest_alive            # (B,Ec,Ec) legal source & alive dest
    n = torch.floor(raw) * gate
    n = torch.where(n >= float(MIN_LAUNCH_SHIPS), n, torch.zeros_like(n))   # drop dribbles -> stay home
    can = (n >= 1.0).to(DTYPE)
    # intercept heading for every (src,dst): src = row planet, dst = col planet
    spx = env.p_x.unsqueeze(2); spy = env.p_y.unsqueeze(2)      # (B,Ec,1) source
    dpx = env.p_x.unsqueeze(1); dpy = env.p_y.unsqueeze(1)      # (B,1,Ec) dest
    off = env.p_radius.unsqueeze(2) + 0.1                       # (B,Ec,1) source surface
    if LEAD_TARGET:
        rdx = dpx - CENTER; rdy = dpy - CENTER
        r_d = torch.sqrt(rdx * rdx + rdy * rdy)                 # (B,1,Ec)
        phi0 = torch.atan2(rdy, rdx)
        drot = (env.p_rotates > 0.5).unsqueeze(1)               # (B,1,Ec)
        w = torch.where(drot, env.ang_vel.view(B, 1, 1), torch.zeros(B, 1, 1, device=dev, dtype=DTYPE))
        v = fleet_speed_t(n.clamp_min(1.0), env.vmax)           # (B,Ec,Ec)
        t = (torch.sqrt((dpx - spx) ** 2 + (dpy - spy) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
        for _ in range(16):
            phi = phi0 + w * t
            ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
            t = (torch.sqrt((ix - spx) ** 2 + (iy - spy) ** 2) - off).clamp_min(0.0) / v.clamp_min(1e-6)
        phi = phi0 + w * t
        ix = CENTER + r_d * torch.cos(phi); iy = CENTER + r_d * torch.sin(phi)
        angle = torch.atan2(iy - spy, ix - spx)                 # (B,Ec,Ec)
    else:
        angle = torch.atan2(dpy - spy, dpx - spx).expand(B, Ec, Ec).contiguous()
    valid = can.sum((1, 2))                                     # launches all go to alive dests
    invalid = torch.zeros(B, dtype=DTYPE, device=dev)           # below-min launches just stay home (no penalty); budget is structural
    launches = can.sum((1, 2))
    return angle, n, can, valid, invalid, launches


def spawn_comets(env):
    '''Spawn COMET_PAIRS point-symmetric comet pairs into free planet slots: neutral objects that
    sweep the inner region at COMET_SPEED and expire past COMET_EXPIRE_RADIUS. Capturable; produce
    COMET_PRODUCTION once owned. Velocity aims past the center at a random perihelion (avoids the sun).'''
    B, dev = env.B, env.dev
    for _pair in range(COMET_PAIRS):
        theta = torch.rand(B, device=dev) * (2.0 * PI)
        d_peri = COMET_PERI_MIN + torch.rand(B, device=dev) * (COMET_PERI_MAX - COMET_PERI_MIN)
        sgn = torch.where(torch.rand(B, device=dev) < 0.5, torch.ones(B, device=dev), -torch.ones(B, device=dev))
        ships = torch.randint(1, 100, (B, 4), device=dev).min(1).values.to(DTYPE)   # min-of-4 -> skewed low
        for _c in range(2):
            th = theta + (0.0 if _c == 0 else PI)                       # point-symmetric pair
            r0 = float(COMET_SPAWN_RADIUS)
            px = CENTER + r0 * torch.cos(th); py = CENTER + r0 * torch.sin(th)
            toward = torch.atan2(CENTER - py, CENTER - px)
            alpha = torch.asin((d_peri / r0).clamp(max=0.99)) * (sgn if _c == 0 else -sgn)
            vdir = toward + alpha
            vx = COMET_SPEED * torch.cos(vdir); vy = COMET_SPEED * torch.sin(vdir)
            free = env.p_alive < 0.5
            slot = torch.argmax(free.to(torch.int8), 1, keepdim=True)    # (B,1) first free slot
            has = free.any(1, keepdim=True)
            def _set(field, val):
                v = val.unsqueeze(1) if val.dim() == 1 else val
                field.scatter_(1, slot, torch.where(has, v, field.gather(1, slot)))
            _set(env.p_alive, torch.ones(B, device=dev))
            _set(env.p_owner, torch.full((B,), -1.0, device=dev))
            _set(env.p_x, px); _set(env.p_y, py); _set(env.p_init_x, px); _set(env.p_init_y, py)
            _set(env.p_comet_vx, vx); _set(env.p_comet_vy, vy)
            _set(env.p_ships, ships)
            _set(env.p_prod, torch.full((B,), float(COMET_PRODUCTION), device=dev))
            _set(env.p_radius, torch.full((B,), float(COMET_RADIUS), device=dev))
            _set(env.p_is_comet, torch.ones(B, device=dev))
            _set(env.p_rotates, torch.zeros(B, device=dev))


def spawn_comets_official(env, e):
    """Place the 4 precomputed symmetric comets of spawn event e at waypoint 0 (free planet slots)."""
    B, dev = env.B, env.dev
    has = env.c_len[:, e] > 0                                    # (B,) event exists for this world
    one = torch.ones(B, 1, dtype=DTYPE, device=dev)
    for m in range(4):
        free = env.p_alive < 0.5
        slot = torch.argmax(free.to(torch.int8), 1, keepdim=True)            # (B,1) first free slot
        ok = (has & free.any(1)).unsqueeze(1)                                # (B,1)
        px = env.c_paths[:, e, m, 0, 0:1]; py = env.c_paths[:, e, m, 0, 1:2]
        def _set(field, val):
            field.scatter_(1, slot, torch.where(ok, val, field.gather(1, slot)))
        _set(env.p_alive, one)
        _set(env.p_owner, -one)
        _set(env.p_x, px); _set(env.p_y, py)
        _set(env.p_init_x, px); _set(env.p_init_y, py)
        _set(env.p_ships, env.c_ships[:, e].unsqueeze(1))
        _set(env.p_prod, one * float(COMET_PRODUCTION))
        _set(env.p_radius, one * float(COMET_RADIUS))
        _set(env.p_is_comet, one)
        _set(env.p_rotates, torch.zeros_like(one))
        env.c_slot[:, e, m] = torch.where(ok.squeeze(1), slot.squeeze(1),
                                          torch.full_like(slot.squeeze(1), -1))


def env_step(env, ego_action, opponent=None, opp_action=None, step_idx=None, seats=None):
    """One tick, N-player (v8). ego_action (B,Ec,Ec+1)=[alloc | fire] for player 0.
    seats = list of opponent seat dicts, one per pid 1..n_players-1:
        {"pid": p, "script": code}    scripted bot (opponent_action codes), or
        {"pid": p, "action": tensor}  neural (B,Ec,Ec+1) gated-alloc action.
    v7 back-compat: seats=None -> a single pid-1 seat built from (opponent, opp_action)."""
    B, Ec, Fc = env.B, env.Ec, env.Fc
    N = int(getattr(env, "n_players", 2))
    if seats is None:
        seats = [{"pid": 1, "action": opp_action}] if opp_action is not None else [{"pid": 1, "script": opponent}]
    _sc = int(env.step_ct[0].item()) if step_idx is None else int(step_idx)   # avoid a per-step CUDA sync
    if COMETS_ENABLED and _sc in COMET_SPAWN_STEPS:   # hidden-schedule comet spawn
        if COMET_OFFICIAL and getattr(env, 'c_paths', None) is not None:
            spawn_comets_official(env, COMET_SPAWN_STEPS.index(_sc))
        else:
            spawn_comets(env)
    vmax = env.vmax
    dev = env.dev
    ego = 0

    ego_owned0 = (env.p_owner == float(ego)) & (env.p_alive > 0.5)
    comet0 = env.p_is_comet > 0.5   # comet status at step start (mask comet expiry out of capture/lost)

    # --- decode ego action (gated allocation: fire -> full garrison, routed) ---
    legal = (env.p_owner == float(ego)) & (env.p_alive > 0.5) & (env.p_ships > 0.0)
    e_ang, e_shp, e_can, valid, invalid, launches = _decode_target(env, ego_action, legal)

    # --- opponent seat launches (scripted: one launch/planet; neural: full alloc decode) ---
    slot_idx = torch.arange(Ec, dtype=torch.long, device=dev).unsqueeze(0).expand(B, Ec)
    src_mat = torch.arange(Ec, dtype=torch.long, device=dev).view(1, Ec, 1).expand(B, Ec, Ec).reshape(B, Ec * Ec)
    flatM = lambda x: x.reshape(B, Ec * Ec)                                   # (B,Ec,Ec) -> (B,Ec*Ec)
    own_blk = [torch.full((B, Ec * Ec), float(ego), dtype=DTYPE, device=dev)]
    slot_blk = [src_mat]
    ang_blk = [flatM(e_ang)]; shp_blk = [flatM(e_shp)]; can_blk = [flatM(e_can)]
    ded = e_shp.sum(2)                                                        # (B,Ec) ships deducted/source
    for st in seats:
        pid = int(st["pid"])
        if st.get("action") is not None:          # neural seat: decode exactly like the ego
            legal_p = (env.p_owner == float(pid)) & (env.p_alive > 0.5) & (env.p_ships > 0.0)
            o_ang, o_shp, o_can, _, _, _ = _decode_target(env, st["action"], legal_p)
            ded = ded + o_shp.sum(2)
            own_blk.append(torch.full((B, Ec * Ec), float(pid), dtype=DTYPE, device=dev))
            slot_blk.append(src_mat)
            ang_blk.append(flatM(o_ang)); shp_blk.append(flatM(o_shp)); can_blk.append(flatM(o_can))
        else:                                     # scripted seat: one launch per owned planet
            o_ang, o_shp, o_can = opponent_action(env, int(st["script"]), pid)
            ded = ded + o_shp
            own_blk.append(torch.full((B, Ec), float(pid), dtype=DTYPE, device=dev))
            slot_blk.append(slot_idx)
            ang_blk.append(o_ang); shp_blk.append(o_shp); can_blk.append(o_can)

    # --- deduct ships from origin planets ---
    env.p_ships = env.p_ships - ded

    # --- place fleets (ego first, then seats in pid order) ---
    owner = torch.cat(own_blk, 1)
    from_slot = torch.cat(slot_blk, 1)
    angle = torch.cat(ang_blk, 1)
    ships = torch.cat(shp_blk, 1)
    commit = torch.cat(can_blk, 1)
    Ltot = owner.shape[1]
    seq = (env.step_ct * float(Ltot + 1)).unsqueeze(1) + \
          torch.arange(Ltot, dtype=DTYPE, device=dev).unsqueeze(0)
    launch_fleets(env, owner, from_slot, angle, ships, commit, seq)

    # --- production ---
    env.p_ships = env.p_ships + env.p_prod * (env.p_owner != -1.0).to(DTYPE) * (env.p_alive > 0.5).to(DTYPE)

    # --- planet new positions (orbit) ---
    stepf = env.step_ct
    dxc = env.p_init_x - CENTER; dyc = env.p_init_y - CENTER
    r = torch.sqrt(dxc * dxc + dyc * dyc)
    ia = torch.atan2(dyc, dxc)
    ca = ia + env.ang_vel.unsqueeze(1) * stepf.unsqueeze(1)
    rot = env.p_rotates > 0.5
    cmt = env.p_is_comet > 0.5
    nx = torch.where(rot, CENTER + r * torch.cos(ca), torch.where(cmt, env.p_x + env.p_comet_vx, env.p_x))
    ny = torch.where(rot, CENTER + r * torch.sin(ca), torch.where(cmt, env.p_y + env.p_comet_vy, env.p_y))
    old_px, old_py = env.p_x, env.p_y
    _comet_expired = None
    if COMETS_ENABLED and COMET_OFFICIAL and getattr(env, 'c_paths', None) is not None:
        # waypoint playback: comet of event e at tick u sweeps path[u-s] -> path[u-s+1]
        _comet_expired = torch.zeros(B, Ec, dtype=torch.bool, device=dev)
        _ar = torch.arange(B, device=dev)
        for _e, _s_e in enumerate(COMET_SPAWN_STEPS):
            if not (_s_e <= _sc <= _s_e + COMET_MAX_LEN):   # at most one event is ever live
                continue
            k = int(_sc - _s_e + 1)                          # waypoint index this tick (same for all envs)
            for _m in range(4):
                sl = env.c_slot[:, _e, _m]                   # (B,) planet slot, -1 = none
                live = sl >= 0
                if not bool(live.any()):
                    continue
                slc = sl.clamp_min(0).unsqueeze(1)
                adv = live & (k < env.c_len[:, _e])          # still has waypoints -> advance
                exp = live & (k >= env.c_len[:, _e])         # path ended -> expire after combat
                wp = env.c_paths[_ar, _e, _m, min(k, COMET_MAX_LEN - 1)]   # (B,2)
                nx.scatter_(1, slc, torch.where(adv.unsqueeze(1), wp[:, 0:1], nx.gather(1, slc)))
                ny.scatter_(1, slc, torch.where(adv.unsqueeze(1), wp[:, 1:2], ny.gather(1, slc)))
                _comet_expired.scatter_(1, slc, exp.unsqueeze(1) | _comet_expired.gather(1, slc))
                env.c_slot[:, _e, _m] = torch.where(exp, torch.full_like(sl, -1), sl)

    # --- fleet movement + swept collision against planet paths ---
    falive = env.f_alive > 0.5
    speed = fleet_speed_t(env.f_ships, vmax)
    fox, foy = env.f_x, env.f_y
    fnx = fox + torch.cos(env.f_angle) * speed
    fny = foy + torch.sin(env.f_angle) * speed
    Ax = fox.unsqueeze(2); Ay = foy.unsqueeze(2)
    Bx = fnx.unsqueeze(2); By = fny.unsqueeze(2)
    P0x = old_px.unsqueeze(1); P0y = old_py.unsqueeze(1)
    P1x = nx.unsqueeze(1); P1y = ny.unsqueeze(1)
    rad = env.p_radius.unsqueeze(1)
    palive = (env.p_alive.unsqueeze(1) > 0.5)
    d0x = Ax - P0x; d0y = Ay - P0y
    dvx = (Bx - Ax) - (P1x - P0x); dvy = (By - Ay) - (P1y - P0y)
    a = dvx * dvx + dvy * dvy
    b = 2.0 * (d0x * dvx + d0y * dvy)
    c = d0x * d0x + d0y * d0y - rad * rad
    disc = b * b - 4.0 * a * c
    sq = torch.sqrt(disc.clamp_min(0.0))
    t1 = (-b - sq) / (2.0 * a)
    t2 = (-b + sq) / (2.0 * a)
    hit_quad = (disc >= 0.0) & (t2 >= 0.0) & (t1 <= 1.0)
    hit_lin = (a < 1e-12) & (c <= 0.0)
    hit = torch.where(a < 1e-12, hit_lin, hit_quad)
    hit = hit & palive & falive.unsqueeze(2)
    slotf = torch.arange(Ec, dtype=DTYPE, device=dev).view(1, 1, Ec)
    order = torch.where(hit, slotf, torch.full_like(slotf, float(Ec)))
    fh_v, tgt_slot = order.min(2)
    has_hit = fh_v < float(Ec)

    # OOB / sun removal (point-to-segment to sun center)
    oob = (fnx < 0.0) | (fnx > BOARD_SIZE) | (fny < 0.0) | (fny > BOARD_SIZE)
    vx, vy, wx, wy = fox, foy, fnx, fny
    l2 = (vx - wx) ** 2 + (vy - wy) ** 2
    tt = ((CENTER - vx) * (wx - vx) + (CENTER - vy) * (wy - vy)) / l2.clamp_min(1e-12)
    tt = tt.clamp(0.0, 1.0)
    prx = vx + tt * (wx - vx); pry = vy + tt * (wy - vy)
    sundist = torch.sqrt((CENTER - prx) ** 2 + (CENTER - pry) ** 2)
    sun_hit = (l2 > 0.0) & (sundist < SUN_RADIUS)
    sun_pt = torch.sqrt((CENTER - vx) ** 2 + (CENTER - vy) ** 2) < SUN_RADIUS
    sun_hit = torch.where(l2 > 0.0, sun_hit, sun_pt)

    remove_fleet = falive & (has_hit | oob | sun_hit)
    contributes = falive & has_hit

    # --- combat: arrivals per PLAYER, official top-vs-second resolve (N-player) ---
    cf = contributes.to(DTYPE)
    tslot = tgt_slot.clamp(0, Ec - 1)
    arr_p = []
    for p in range(N):
        sp = env.f_ships * cf * (env.f_owner == float(p)).to(DTYPE)
        ap = torch.zeros(B, Ec, dtype=DTYPE, device=dev)
        ap.scatter_add_(1, tslot, sp)
        arr_p.append(ap)
    arr = torch.stack(arr_p, 2)                                  # (B,Ec,N)
    top2v, top2i = arr.topk(2, dim=2)                            # N >= 2 always
    surv_ships = top2v[..., 0] - top2v[..., 1]                   # exact tie -> 0 survivors
    surv_owner = torch.where(surv_ships > 0.0, top2i[..., 0].to(DTYPE),
                             torch.full((B, Ec), -1.0, dtype=DTYPE, device=dev))
    any_arr = top2v[..., 0] > 0.0
    apply = any_arr & (surv_ships > 0.0) & (env.p_alive > 0.5)
    same = (env.p_owner == surv_owner)
    reinforce = apply & same
    attack = apply & (~same)
    env.p_ships = torch.where(reinforce, env.p_ships + surv_ships, env.p_ships)
    after = env.p_ships - surv_ships
    flips = attack & (after < 0.0)
    env.p_ships = torch.where(attack, torch.where(after < 0.0, -after, after), env.p_ships)
    env.p_owner = torch.where(flips, surv_owner, env.p_owner)

    # --- apply planet positions; clear removed fleets ---
    env.p_x = nx; env.p_y = ny
    if COMETS_ENABLED:                                   # comet expiry: official = path end; legacy = swept past the inner region
        if _comet_expired is not None:
            gone = _comet_expired & (env.p_alive > 0.5)
        else:
            cdist = torch.sqrt((env.p_x - CENTER) ** 2 + (env.p_y - CENTER) ** 2)
            gone = (env.p_is_comet > 0.5) & (env.p_alive > 0.5) & (cdist > COMET_EXPIRE_RADIUS)
        keepc = (~gone).to(DTYPE)
        env.p_alive = env.p_alive * keepc; env.p_is_comet = env.p_is_comet * keepc
        env.p_comet_vx = env.p_comet_vx * keepc; env.p_comet_vy = env.p_comet_vy * keepc
        env.p_ships = env.p_ships * keepc
        env.p_owner = torch.where(gone, torch.full_like(env.p_owner, -1.0), env.p_owner)
    keep = falive & (~remove_fleet)
    keepf = keep.to(DTYPE)
    env.f_alive = keepf
    env.f_owner = env.f_owner * keepf
    env.f_x = fnx * keepf; env.f_y = fny * keepf
    env.f_angle = env.f_angle * keepf
    env.f_ships = env.f_ships * keepf

    env.step_ct = env.step_ct + 1.0

    ego_owned1 = (env.p_owner == float(ego)) & (env.p_alive > 0.5)

    out = StepOut()
    out.invalid = invalid
    out.valid = valid
    out.launches = launches
    out.launched_ships = e_shp.sum((1, 2))   # total ships launched this step (ship-weighted launch reward)
    out.owned_launch_ships = (e_shp * ego_owned0.unsqueeze(1).to(DTYPE)).sum((1, 2))   # ships launched to ALREADY-OWNED planets
    noncomet1 = env.p_is_comet < 0.5
    out.captured = (ego_owned1 & (~ego_owned0) & noncomet1).to(DTYPE).sum(1)
    out.lost = (ego_owned0 & (~ego_owned1) & (~comet0)).to(DTYPE).sum(1)
    out.captured_prod = (env.p_prod * (ego_owned1 & (~ego_owned0) & noncomet1).to(DTYPE)).sum(1)
    out.lost_prod = (env.p_prod * (ego_owned0 & (~ego_owned1) & (~comet0)).to(DTYPE)).sum(1)
    return out


## 10. Policy net -- v5 target heads (mirrors `model/policy_net.cpp`)
Same trunk; heads swapped to `dest_head` (B,E,E categorical logits), `phi_mu` (B,E,1),
`phi_logstd` (B,E,1). `dest_head` out dim == `PLANET_CAP` (the dest is an obs-slot index).

In [ ]:
import contextlib
def _amp_ctx():
    if USE_AMP and DEVICE.type == 'cuda':
        return torch.autocast('cuda', dtype=AMP_DTYPE)
    return contextlib.nullcontext()

class PopArt:
    '''Output-preserving running normalization of value targets (Hessel et al. 2018). The value head
    predicts NORMALIZED values; raw V = denormalize(y); on each stats update val_out is rescaled so
    the raw predictions are preserved.'''
    def __init__(self, beta=POPART_BETA):
        self.mu = 0.0; self.nu = 1.0; self.sigma = 1.0; self.beta = beta; self.init = False
    def normalize(self, y):
        return (y - self.mu) / self.sigma
    def denormalize(self, yn):
        return yn * self.sigma + self.mu
    def update(self, targets, val_out):
        with torch.no_grad():
            old_mu, old_sigma = self.mu, self.sigma
            m = targets.mean().item(); v = (targets * targets).mean().item()
            if not self.init:
                self.mu, self.nu, self.init = m, v, True
            else:
                b = self.beta
                self.mu = (1 - b) * self.mu + b * m
                self.nu = (1 - b) * self.nu + b * v
            self.sigma = max((self.nu - self.mu * self.mu) ** 0.5, 1e-4)
            if old_sigma > 0:                       # preserve raw outputs of the value head
                val_out.weight.mul_(old_sigma / self.sigma)
                val_out.bias.copy_((old_sigma * val_out.bias + old_mu - self.mu) / self.sigma)

class TxEncoderLayer(nn.Module):
    '''Pre-LN transformer encoder block: x = x + MHA(LN(x)); x = x + MLP(LN(x)).
    Multi-head; masks dead-planet KEYS (matches the legacy single-head attention).'''
    def __init__(self, h, n_heads, mlp_ratio):
        super().__init__()
        assert h % n_heads == 0, "HIDDEN (%d) not divisible by N_HEADS (%d)" % (h, n_heads)
        self.h, self.nh, self.hd = h, n_heads, h // n_heads
        self.ln1 = nn.LayerNorm(h)
        self.q = nn.Linear(h, h); self.k = nn.Linear(h, h); self.v = nn.Linear(h, h); self.o = nn.Linear(h, h)
        self.ln2 = nn.LayerNorm(h)
        self.mlp_in = nn.Linear(h, mlp_ratio * h); self.mlp_out = nn.Linear(mlp_ratio * h, h)
    def forward(self, x, entity_mask):
        B, E, _ = x.shape
        xn = self.ln1(x)
        q = self.q(xn).view(B, E, self.nh, self.hd).transpose(1, 2)        # (B,nh,E,hd)
        k = self.k(xn).view(B, E, self.nh, self.hd).transpose(1, 2)
        v = self.v(xn).view(B, E, self.nh, self.hd).transpose(1, 2)
        attn_mask = (entity_mask < 0.5).view(B, 1, 1, E).to(q.dtype) * (-1e9)   # additive key mask
        ctx = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)      # flash/mem-efficient (B,nh,E,hd)
        ctx = ctx.transpose(1, 2).reshape(B, E, self.h)
        x = x + self.o(ctx)
        x = x + self.mlp_out(torch.relu(self.mlp_in(self.ln2(x))))
        return x


class TransformerTrunk(nn.Module):
    '''proj -> n_stem pre-LN ResNet-MLP -> n_layers transformer blocks -> n_head pre-LN
    ResNet-MLP -> LN. Emits per-planet features (B,E,h); drop-in for the legacy trunk.'''
    def __init__(self, F, h, n_heads, n_layers, mlp_ratio, n_stem, n_head):
        super().__init__()
        self.proj = nn.Linear(F, h)
        self.stem_a = nn.ModuleList([nn.Linear(h, h) for _ in range(n_stem)])
        self.stem_b = nn.ModuleList([nn.Linear(h, h) for _ in range(n_stem)])
        self.stem_ln = nn.ModuleList([nn.LayerNorm(h) for _ in range(n_stem)])
        self.layers = nn.ModuleList([TxEncoderLayer(h, n_heads, mlp_ratio) for _ in range(n_layers)])
        self.head_a = nn.ModuleList([nn.Linear(h, h) for _ in range(n_head)])
        self.head_b = nn.ModuleList([nn.Linear(h, h) for _ in range(n_head)])
        self.head_ln = nn.ModuleList([nn.LayerNorm(h) for _ in range(n_head)])
        self.ln_out = nn.LayerNorm(h)
    def forward(self, entities, entity_mask):
        tok = self.proj(entities)
        for i in range(len(self.stem_a)):
            tok = tok + self.stem_b[i](torch.relu(self.stem_a[i](self.stem_ln[i](tok))))
        for layer in self.layers:
            tok = layer(tok, entity_mask)
        for i in range(len(self.head_a)):
            tok = tok + self.head_b[i](torch.relu(self.head_a[i](self.head_ln[i](tok))))
        return self.ln_out(tok)


class PolicyNet(nn.Module):
    '''v5 TARGET actor-critic (mirrors model/policy_net.cpp, target_actor branch).

    Trunk (UNCHANGED from the continuous actor): per-planet proj + masked cross-planet
    self-attention + GLU + n residual blocks + a separate board-globals embedding.
    Heads: dest_head -> (B,E,E) per-source WHERE allocation logits (-> Dirichlet rows over the
    E=PLANET_CAP dests; dest_head out dim == max_entities); gate_head -> (B,E,1) per-source launch
    gate logit. See GatedAllocDist for how WHERE x GATE compose into the (B,E,E+1) action.
    '''
    def __init__(self, F, G, hidden, d_g, max_entities, n_res_blocks, use_glu, init_gate_bias,
                 use_attention=True,
                 value_res_blocks=2, arch="trunk", n_heads=8, n_tx_layers=4, tx_mlp_ratio=4,
                 n_stem_res=1, n_head_res=1):
        super().__init__()
        self.F, self.G, self.h, self.d_g, self.E = F, G, hidden, d_g, max_entities
        self.use_glu = use_glu
        self.use_attention = use_attention
        self.arch = arch
        h = hidden
        if arch == "transformer":
            self.trunk = TransformerTrunk(F, h, n_heads, n_tx_layers, tx_mlp_ratio, n_stem_res, n_head_res)
        else:
            self.proj = nn.Linear(F, h)
            if use_attention:                               # cross-planet self-attention (toggleable)
                self.attn_q = nn.Linear(h, h); self.attn_k = nn.Linear(h, h)
                self.attn_v = nn.Linear(h, h); self.attn_o = nn.Linear(h, h)
                self.ln_attn = nn.LayerNorm(h)
            self.ln_out = nn.LayerNorm(h)
            if use_glu:
                self.glu_gate = nn.Linear(h, h); self.glu_val = nn.Linear(h, h); self.glu_out = nn.Linear(h, h)
            self.res_a = nn.ModuleList([nn.Linear(h, h) for _ in range(n_res_blocks)])
            self.res_b = nn.ModuleList([nn.Linear(h, h) for _ in range(n_res_blocks)])
            self.ln_res = nn.ModuleList([nn.LayerNorm(h) for _ in range(n_res_blocks)])
        self.g_embed = nn.Linear(G, d_g)
        # --- gated allocation heads: WHERE (per-source dest simplex) + GATE (per-source fire logit) ---
        self.dest_mode = DEST_HEAD
        if DEST_HEAD == 'bilinear':                        # pointer score: query=source, key=dest
            self.dest_q = nn.Linear(h + d_g, h); self.dest_k = nn.Linear(h + d_g, h)
        else:
            self.dest_head = nn.Linear(h + d_g, max_entities)  # (B,E,E) per-source WHERE allocation logits
        self.gate_head = nn.Linear(h + d_g, 1)             # (B,E,1) per-source launch-gate logit
        # value head (PPO critic): input proj -> pre-LN RESIDUAL MLP (same block as the trunk) -> readout
        self.val_in = nn.Linear(h + d_g, h)
        self.val_res_a = nn.ModuleList([nn.Linear(h, h) for _ in range(value_res_blocks)])
        self.val_res_b = nn.ModuleList([nn.Linear(h, h) for _ in range(value_res_blocks)])
        self.val_ln = nn.ModuleList([nn.LayerNorm(h) for _ in range(value_res_blocks)])
        self.val_out = nn.Linear(h, 1)
        # calm init: few planets fire at start (negative gate bias); WHERE small-init -> ~uniform
        with torch.no_grad():
            self.gate_head.bias.fill_(init_gate_bias)
            if DEST_HEAD == 'bilinear':
                self.dest_q.weight.mul_(INIT_DEST_SCALE); self.dest_q.bias.zero_()
                self.dest_k.weight.mul_(INIT_DEST_SCALE); self.dest_k.bias.zero_()
            else:
                self.dest_head.weight.mul_(INIT_DEST_SCALE); self.dest_head.bias.zero_()

    def forward(self, entities, entity_mask, action_mask, globals_):
        if self.arch == "transformer":
            tok = self.trunk(entities, entity_mask)
        else:
            tok = self.proj(entities)                       # (B,E,d)
            # masked single-head self-attention over planets (pre-norm) -- the only cross-planet mixing
            if self.use_attention:
                xn = self.ln_attn(tok)
                Q = self.attn_q(xn); Kt = self.attn_k(xn); Vt = self.attn_v(xn)
                scores = torch.matmul(Q, Kt.transpose(1, 2)) / math.sqrt(self.h)   # (B,E,E)
                dead_key = (entity_mask < 0.5).unsqueeze(1)     # (B,1,E)
                scores = scores.masked_fill(dead_key, -1e9)
                attn = torch.matmul(torch.softmax(scores, -1), Vt)
                tok = tok + self.attn_o(attn)
            if self.use_glu:
                tok = tok + self.glu_out(self.glu_val(tok) * torch.sigmoid(self.glu_gate(tok)))
            for i in range(len(self.res_a)):
                tok = tok + self.res_b[i](torch.relu(self.res_a[i](self.ln_res[i](tok))))
            tok = self.ln_out(tok)
        B, E = tok.shape[0], tok.shape[1]
        gp = torch.relu(self.g_embed(globals_))         # (B,d_g)
        gpb = gp.unsqueeze(1).expand(B, E, self.d_g)
        hh = torch.cat([tok, gpb], -1)                  # (B,E,d+d_g)
        # --- v5 target heads ---
        if self.dest_mode == 'bilinear':               # pointer score q.k / sqrt(h)
            qd = self.dest_q(hh); kd = self.dest_k(hh)
            dest_logits = torch.einsum('bsh,bdh->bsd', qd, kd) / math.sqrt(self.h)
        else:
            dest_logits = self.dest_head(hh)           # (B,E,E) per-source WHERE allocation logits
        gate_logits = self.gate_head(hh).squeeze(-1)   # (B,E) per-source launch-gate logit
        # value: masked-mean pool of trunk tokens over live planets + g'
        em = entity_mask.unsqueeze(-1)
        pooled = (tok * em).sum(1) / em.sum(1).clamp_min(1.0)
        vh = self.val_in(torch.cat([pooled, gp], -1))                  # (B,d)
        for i in range(len(self.val_res_a)):                           # residual blocks (skip per block)
            vh = vh + self.val_res_b[i](torch.relu(self.val_res_a[i](self.val_ln[i](vh))))
        value = self.val_out(vh).squeeze(-1)                           # (B,)
        return dest_logits, gate_logits, value


def build_policy():
    _net = PolicyNet(F_DIM, G_DIM, HIDDEN, D_G, PLANET_CAP, N_RES_BLOCKS, USE_GLU, INIT_GATE_BIAS,
                     use_attention=USE_ATTENTION,
                     value_res_blocks=VALUE_RES_BLOCKS,
                     arch=ARCH, n_heads=N_HEADS, n_tx_layers=N_TX_LAYERS, tx_mlp_ratio=TX_MLP_RATIO,
                     n_stem_res=N_STEM_RES, n_head_res=N_HEAD_RES).to(DEVICE)
    _net.popart = PopArt()
    return _net


def compute_reach(ent, em):
    """Per-source destination reachability (B,E,E): 1 unless a straight src->dest shot is absorbed
    by the sun. Recovered from encoded planet positions so it is identical in rollout and update.
    Diagonal forced to 1; the alive-mask is applied in the distribution."""
    px = ent[..., 0] * BOARD_SIZE
    py = ent[..., 1] * BOARD_SIZE
    B, E = px.shape
    sx = px.unsqueeze(2); sy = py.unsqueeze(2)         # source (B,E,1)
    dx = px.unsqueeze(1) - sx                           # src->dest (B,E,E)
    dy = py.unsqueeze(1) - sy
    dist = torch.sqrt(dx * dx + dy * dy).clamp_min(1e-6)
    ux = dx / dist; uy = dy / dist
    cx = CENTER - sx; cy = CENTER - sy                  # source -> sun centre
    t = cx * ux + cy * uy                               # closest-approach distance along the ray
    perp2 = (cx * cx + cy * cy) - t * t
    sr = SUN_RADIUS * SUN_RADIUS
    t_sun = t - torch.sqrt((sr - perp2).clamp_min(0.0))
    blocked = (t >= 0.0) & (perp2 <= sr) & (t_sun >= 0.0) & (t_sun < dist)
    reach = (~blocked).to(DTYPE)
    eye = torch.eye(E, device=ent.device, dtype=DTYPE).unsqueeze(0)
    return reach * (1.0 - eye) + eye                    # source -> itself is always reachable


def recover_ships(ent):
    """Integer ship count per planet, recovered from feature b6 = log1p(ships)/log(1000)."""
    return torch.round(torch.expm1((ent[..., 6] * SHIP_LOG_DENOM).clamp_min(0.0)))


def _make_dist(net, ent, em, am, gl):
    '''Build the gated-allocation actor distribution. owned = legal source planets (= action_mask),
    alive = valid destination slots (= entity_mask), reach = sun-reachability mask.'''
    reach = compute_reach(ent, em) if REACH_MASK else None
    with _amp_ctx():
        dest_logits, gate_logits, value = net(ent, em, am, gl)
    if WHERE_DIST == 'categorical':
        dist = GatedCatDist(dest_logits.float(), gate_logits.float(), owned=am, alive=em, reach=reach)
    else:
        dist = GatedAllocDist(dest_logits.float(), gate_logits.float(), owned=am, alive=em, reach=reach, kappa=ALLOC_KAPPA)
    return dist, value.float()


@torch.no_grad()
def act(net, ent, em, am, gl, greedy=False):
    dist, value = _make_dist(net, ent, em, am, gl)
    action = dist.greedy() if greedy else dist.sample()
    if USE_POPART and getattr(net, 'popart', None) is not None:
        value = net.popart.denormalize(value)
    return action, dist.log_prob(action), value


def evaluate(net, ent, em, am, gl, action):
    dist, value = _make_dist(net, ent, em, am, gl)
    return dist.log_prob(action), dist.entropy(), value

## 11. Rollout + v5 reward assembly + GAE (mirrors `rollout.cpp::collect_gpu`)
v5 reward channels: capture +/-30, first-50 VALID-launch +1, production-milestone +20
per ego total-ship doubling; outcome (win/loss decayed past step 100) scattered onto
the last active step. GAE; reward+outcome scaled by 1/100. Heartbeat logs R[o c d p].

**v8**: `collect_ppo(env, net, pool, cursor, rng, seats, n_players)` -- the learner is always player 0; `stats['seat_scores']` carries the per-seat pairwise scores that feed the dual-Elo update. 4p outcome is placement-linear by default (`OUTCOME_4P`).


In [ ]:
def settle_n(env):
    """Per-player settle: ships (B,N) = planets+fleets per player; alive (B,N) bool."""
    N = int(getattr(env, "n_players", 2))
    pa = env.p_alive > 0.5; fa = env.f_alive > 0.5
    s, alive = [], []
    for p in range(N):
        op = (env.p_owner == float(p)) & pa
        gp = (env.f_owner == float(p)) & fa
        s.append((env.p_ships * op.to(DTYPE)).sum(1) + (env.f_ships * gp.to(DTYPE)).sum(1))
        alive.append(op.any(1) | gp.any(1))
    return torch.stack(s, 1), torch.stack(alive, 1)


def settle(env):
    """v7-compat 2p view: (s0, s1, side0_alive, side1_alive). BC/eval/probe cells use this."""
    s, alive = settle_n(env)
    return s[:, 0], s[:, 1], alive[:, 0], alive[:, 1]


def _potentials(env):
    """Policy-invariant shaping potentials (Ng et al.), N-player: ship-margin and production
    share vs the STRONGEST opponent (reduces exactly to the v7 pair when n_players == 2)."""
    s, _ = settle_n(env)
    phi_ship = ship_log_t(s[:, 0]) - ship_log_t(s[:, 1:].max(1).values)
    pa = env.p_alive > 0.5
    N = int(getattr(env, "n_players", 2))
    prods = [(env.p_prod * ((env.p_owner == float(p)) & pa).to(DTYPE)).sum(1) for p in range(N)]
    pe = prods[0]
    pen = torch.stack(prods[1:], 1).max(1).values
    tot = (env.p_prod * pa.to(DTYPE)).sum(1).clamp_min(1.0)
    return phi_ship, (pe - pen) / tot


_KIND2OPP = {"noop": 2, "random": 0, "starter": 1, "medium": 3, "greedy": 4, "intermediate": 5}


def collect_ppo(env, net, world_pool, cursor, rng, seats, n_players=2, n_envs=None):
    """One PPO iteration (learner = player 0) vs `seats` -- league members filling pids
    1..n_players-1. Scripted anchors run fully on-GPU; neural members act per step (no-grad).
    On-GPU buffers + GPU GAE; reward = potential shaping + outcome. n_players=2 is the v7 path.
    stats["seat_scores"][k] = the learner's mean pairwise score vs seat k (dual-Elo signal)."""
    Bn = int(n_envs or B)
    Ec, T = env.Ec, env.T
    dev = env.dev
    specs = []
    for i, m in enumerate(seats):
        pid = i + 1
        if m.get("net") is None:
            specs.append({"pid": pid, "script": _KIND2OPP[m["kind"]]})
        else:
            specs.append({"pid": pid, "net": m["net"]})

    worlds = [world_pool[(cursor + i) % len(world_pool)] for i in range(Bn)]
    cursor += Bn
    env.reset(worlds, n_players=n_players)

    # on-GPU rollout buffers (compute-bound: no host copies, no per-step D2H sync)
    ent_buf = torch.zeros(T, Bn, Ec, F_DIM, device=dev)
    em_buf = torch.zeros(T, Bn, Ec, device=dev); am_buf = torch.zeros(T, Bn, Ec, device=dev)
    gl_buf = torch.zeros(T, Bn, G_DIM, device=dev); act_buf = torch.zeros(T, Bn, Ec, Ec + 1, device=dev)
    oldlp_buf = torch.zeros(T, Bn, device=dev); valid_buf = torch.zeros(T, Bn, device=dev)
    rew_buf = torch.zeros(T, Bn, device=dev); val_buf = torch.zeros(T, Bn, device=dev); done_buf = torch.zeros(T, Bn, device=dev)

    active = torch.ones(Bn, device=dev); outcome = torch.zeros(Bn, device=dev)
    seat_sc = torch.zeros(len(specs), Bn, device=dev)   # pairwise score vs each seat, frozen at term
    inv_sum = torch.zeros(Bn, device=dev); lnch_sum = torch.zeros(Bn, device=dev)
    valid_sum = torch.zeros(Bn, device=dev); step_sum = torch.zeros(Bn, device=dev)
    launch_paid = torch.zeros(Bn, device=dev); milestone_prev = torch.zeros(Bn, device=dev)
    captureR = torch.zeros(Bn, device=dev); launchR = torch.zeros(Bn, device=dev); prodMR = torch.zeros(Bn, device=dev)

    phi_ship_prev, phi_prod_prev = _potentials(env)   # shaping potentials at s_0

    def _scores_outcome(s_all):
        """Pairwise scores vs every seat + the scalar outcome in [-1, 1]."""
        s0 = s_all[:, 0]
        scs = []
        for sp in specs:
            sk = s_all[:, sp["pid"]]
            scs.append(torch.where(s0 > sk, torch.ones_like(s0),
                                   torch.where(s0 < sk, torch.zeros_like(s0), torch.full_like(s0, 0.5))))
        if n_players == 2 or OUTCOME_4P == "winner":
            oc = torch.sign(s0 - s_all[:, 1:].max(1).values)    # official: top score only
        else:
            oc = 2.0 * torch.stack(scs, 0).mean(0) - 1.0        # placement-linear (beat-all = +1)
        return scs, oc

    last_t = 0
    for t in range(T):
        last_t = t
        ent, em, am, gl = env_encode(env, 0)
        a_t, logp, value = act(net, ent, em, am, gl, greedy=False)
        ent_buf[t] = ent; em_buf[t] = em; am_buf[t] = am; gl_buf[t] = gl
        act_buf[t] = a_t; oldlp_buf[t] = logp; valid_buf[t] = active; val_buf[t] = value

        step_seats = []
        for sp in specs:
            if "net" in sp:
                with torch.no_grad():
                    oe, om, oa, og = env_encode(env, sp["pid"])
                    o_act, _, _ = act(sp["net"], oe, om, oa, og, greedy=False)
                step_seats.append({"pid": sp["pid"], "action": o_act})
            else:
                step_seats.append({"pid": sp["pid"], "script": sp["script"]})
        out = env_step(env, a_t, seats=step_seats, step_idx=t)
        a = active

        if USE_POTENTIAL_SHAPING:
            phi_ship_now, phi_prod_now = _potentials(env)
            cap_t = a * (SHAPE_SHIP * (GAMMA * phi_ship_now - phi_ship_prev))   # ship-margin shaping
            mr_t = a * (SHAPE_PROD * (GAMMA * phi_prod_now - phi_prod_prev))    # production-share shaping
            launch_t = torch.zeros_like(cap_t)
            phi_ship_prev, phi_prod_prev = phi_ship_now, phi_prod_now
        else:
            s_all0, _ = settle_n(env)
            s0n = s_all0[:, 0]
            cap_t = a * (CAPTURE_REWARD * ((out.captured + CAPTURE_PROD_SCALE * out.captured_prod)
                                           - CAPTURE_LOSS_FRAC * (out.lost + CAPTURE_PROD_SCALE * out.lost_prod)))
            m_now = torch.where(s0n >= PROD_MILESTONE_BASE,
                                torch.floor(torch.log2(s0n.clamp_min(PROD_MILESTONE_BASE) / PROD_MILESTONE_BASE)) + 1.0,
                                torch.zeros_like(s0n))
            mr_t = a * (PROD_MILESTONE_REWARD * (m_now - milestone_prev).clamp_min(0.0))
            milestone_prev = torch.maximum(milestone_prev, m_now)
            if t < LAUNCH_WINDOW:
                _enemy_ships = out.launched_ships - out.owned_launch_ships
                _self_ships = torch.clamp(out.owned_launch_ships, max=SELF_LAUNCH_CAP)
                _disp_raw = LAUNCH_REWARD * _enemy_ships + SELF_LAUNCH_REWARD * _self_ships
                launch_t = a * torch.clamp(_disp_raw, max=LAUNCH_STEP_CAP)
            else:
                launch_t = torch.zeros_like(cap_t)
            launch_room = (LAUNCH_GAME_CAP - launch_paid).clamp_min(0.0)
            launch_t = torch.minimum(launch_t, launch_room)
            launch_paid = launch_paid + launch_t
        captureR = captureR + cap_t; prodMR = prodMR + mr_t; launchR = launchR + launch_t
        rew_buf[t] = (cap_t + mr_t + launch_t) / PPO_REWARD_SCALE

        inv_sum = inv_sum + a * out.invalid
        lnch_sum = lnch_sum + a * out.launches
        valid_sum = valid_sum + a * out.valid
        step_sum = step_sum + a

        s_all, alive_all = settle_n(env)
        step_now = env.step_ct
        n_alive = alive_all.to(DTYPE).sum(1)
        term = (step_now >= float(T - 2)) | (n_alive <= 1.0) | (~alive_all[:, 0])   # ego dead -> decided
        done_buf[t] = ((active > 0.5) & term).to(DTYPE)        # terminal on cap OR death (no spurious bootstrap)
        newly = (active > 0.5) & term
        scs, oc = _scores_outcome(s_all)
        for k in range(len(specs)):
            seat_sc[k] = torch.where(newly, scs[k], seat_sc[k])
        outcome = torch.where(newly, oc, outcome)
        active = torch.where(term, torch.zeros_like(active), active)
        # NOTE: no per-step active.sum().item() early-break -> that is a CUDA sync. Finished envs are masked.

    s_all, alive_all = settle_n(env)
    scs, oc = _scores_outcome(s_all)
    still = active > 0.5
    for k in range(len(specs)):
        seat_sc[k] = torch.where(still, scs[k], seat_sc[k])
    outcome = torch.where(still, oc, outcome)

    with torch.no_grad():
        fe, fm, fa_, fg = env_encode(env, 0)
        _, _, vT = act(net, fe, fm, fa_, fg, greedy=False)
        bootstrap = vT * active

    Tu = last_t + 1
    rew = rew_buf[:Tu].clone(); val = val_buf[:Tu]; done = done_buf[:Tu]; alive = valid_buf[:Tu]
    length = alive.sum(0)
    len_eff = (length - DECAY_START_STEP).clamp_min(0.0)
    win_val = torch.pow(torch.full_like(outcome, WIN_DECAY), len_eff) * WIN_BONUS
    loss_val = torch.pow(torch.full_like(outcome, LOSS_DECAY), len_eff) * LOSS_PENALTY
    # fractional outcomes (4p placement mode) scale the outcome magnitudes linearly;
    # in 2p outcome is exactly {-1, 0, +1} -> identical to the v7 payment
    Ot = torch.where(outcome > 0.0, outcome * win_val, outcome * loss_val)
    r_outcome_log = Ot.mean().item()
    if not USE_POTENTIAL_SHAPING:
        disp_cashout = torch.clamp(LAUNCH_WINDOW - length, min=0.0) * LAUNCH_STEP_CAP * (outcome > 0.0).to(outcome.dtype)
        disp_cashout = torch.minimum(disp_cashout, (LAUNCH_GAME_CAP - launch_paid).clamp_min(0.0))
        launchR = launchR + disp_cashout
        Ot = (Ot + disp_cashout) / PPO_REWARD_SCALE
    else:
        Ot = Ot / PPO_REWARD_SCALE
    last_idx = (length - 1.0).clamp_min(0.0).long().unsqueeze(0)
    rew.scatter_add_(0, last_idx, Ot.unsqueeze(0))

    # GAE on GPU (no .cpu(), no synchronize)
    adv = torch.zeros(Tu, Bn, device=dev)
    A = torch.zeros(Bn, device=dev); nextval = bootstrap.clone()
    for t in range(Tu - 1, -1, -1):
        al = alive[t]; notdone = 1.0 - done[t]
        delta = rew[t] + GAMMA * nextval * notdone - val[t]
        A = delta + GAMMA * GAE_LAMBDA * notdone * A
        adv[t] = A * al
        nextval = val[t]
        A = A * al
    ret_full = (adv + val).reshape(-1); adv_full = adv.reshape(-1)
    mean_return_log = (rew.sum(0).mean().item()) * PPO_REWARD_SCALE

    keep = valid_buf[:Tu].reshape(-1).nonzero().squeeze(-1)
    def sel(x): return x.index_select(0, keep)
    tb = {}
    tb["entities"] = sel(ent_buf[:Tu].reshape(Tu * Bn, Ec, F_DIM))
    tb["entity_mask"] = sel(em_buf[:Tu].reshape(Tu * Bn, Ec))
    tb["action_mask"] = sel(am_buf[:Tu].reshape(Tu * Bn, Ec))
    tb["globals"] = sel(gl_buf[:Tu].reshape(Tu * Bn, G_DIM))
    tb["action"] = sel(act_buf[:Tu].reshape(Tu * Bn, Ec, Ec + 1))
    tb["old_logp"] = sel(oldlp_buf[:Tu].reshape(Tu * Bn))
    adv_s = sel(adv_full)
    tb["advantage"] = (adv_s - adv_s.mean()) / (adv_s.std() + 1e-8)
    tb["returns"] = sel(ret_full)
    tb["n"] = keep.shape[0]
    del ent_buf, em_buf, am_buf, gl_buf, act_buf, oldlp_buf, val_buf, rew_buf, done_buf, valid_buf, adv, adv_full, ret_full

    step_total = step_sum.sum().item()
    stats = {
        "mean_return": mean_return_log,
        "mean_len": step_total / Bn,
        "win_rate": (seat_sc.min(0).values >= 0.999).to(DTYPE).mean().item(),   # beat EVERY opponent
        "score": seat_sc.mean().item(),
        "seat_scores": [float(seat_sc[k].mean().item()) for k in range(len(specs))],
        "fmt": n_players,
        "inv_per_step": (inv_sum.sum().item() / step_total) if step_total > 0 else 0.0,
        "lnch_per_step": (lnch_sum.sum().item() / step_total) if step_total > 0 else 0.0,
        "valid_per_step": (valid_sum.sum().item() / step_total) if step_total > 0 else 0.0,
        "transitions": tb["n"],
        "r_outcome": r_outcome_log,
        "r_capture": captureR.mean().item(),      # = ship-margin shaping when USE_POTENTIAL_SHAPING
        "r_launch": launchR.mean().item(),
        "r_milestone": prodMR.mean().item(),      # = production-share shaping when USE_POTENTIAL_SHAPING
    }
    return tb, stats, cursor


## 12. PPO update (mirrors `grpo_trainer.cpp::update`)

Clipped surrogate + `VF_COEF*MSE(value, returns) - ENT_COEF*entropy`; grad clip; skip a
non-finite step. Entropy is **per-component** (`/ (E*3K)`) so `ent_coef` behaves like standard
PPO. Reports approx-KL, clipfrac, sigma.

In [ ]:
_HALF_LOG2PIE_C = 1.4189385332046727

def policy_surrogate(logp, oldlp, adv, clip):
    ratio = torch.exp((logp - oldlp).clamp(-LOGRATIO_CLAMP, LOGRATIO_CLAMP))
    unclipped = ratio * adv
    clipped = torch.clamp(ratio, 1.0 - clip, 1.0 + clip) * adv
    return -torch.minimum(unclipped, clipped).mean()


def ppo_update(net, opt, tb):
    N = tb["n"]; mb = MINIBATCHES; mbsize = N // mb
    s = {"total": 0.0, "policy": 0.0, "vf": 0.0, "entropy": 0.0, "sigma": 0.0,
         "approx_kl": 0.0, "clipfrac": 0.0, "grad_norm": 0.0}
    if mbsize == 0:
        return s
    nsteps = 0
    ent_coef = CUR_ENT_COEF
    dev = tb["entities"].device
    for epoch in range(UPDATE_EPOCHS):
        perm = torch.randperm(N, device=dev)
        ep_kl = []
        for b in range(mb):
            mi = perm[b * mbsize:(b + 1) * mbsize]
            ent = tb["entities"].index_select(0, mi)          # already on GPU (no .to(DEVICE))
            em = tb["entity_mask"].index_select(0, mi)
            am = tb["action_mask"].index_select(0, mi)
            gl = tb["globals"].index_select(0, mi)
            act_mb = tb["action"].index_select(0, mi)
            oldlp = tb["old_logp"].index_select(0, mi)
            adv = tb["advantage"].index_select(0, mi)
            ret = tb["returns"].index_select(0, mi)

            logp, entropy, value = evaluate(net, ent, em, am, gl, act_mb)
            pol = policy_surrogate(logp, oldlp, adv, CLIP)
            vtarget = net.popart.normalize(ret) if (USE_POPART and getattr(net, "popart", None) is not None) else ret
            vloss = F.mse_loss(value, vtarget)
            n_owned = am.sum(1).clamp_min(1.0)                # per-owned-source mean entropy (scale-correct)
            ent_b = (entropy / n_owned).mean()
            pc = globals().get('PPO_POLICY_COEF', 1.0)   # 0 during value warmup
            loss = pc * pol + VF_COEF * vloss - (pc * ent_coef) * ent_b

            opt.zero_grad()
            loss.backward()
            if pc == 0.0:                                  # critic warmup: val-only step --
                for pn, pp in net.named_parameters():      # shared-trunk vf grads otherwise
                    if not pn.startswith('val_'):          # drag the sharp BC policy around
                        pp.grad = None
            gnorm = torch.nn.utils.clip_grad_norm_(net.parameters(), MAX_GRAD_NORM)
            if torch.isfinite(gnorm):
                opt.step()
            if USE_POPART and getattr(net, "popart", None) is not None:
                net.popart.update(ret, net.val_out)           # output-preserving stats update

            with torch.no_grad():
                lr = (logp - oldlp); lrc = lr.clamp(-LOGRATIO_CLAMP, LOGRATIO_CLAMP)
                ratio = torch.exp(lrc)
                mb_kl = ((ratio - 1.0) - lrc).mean().item()   # k3 KL estimator (>=0)
                s["total"] += loss.item(); s["policy"] += pol.item(); s["vf"] += vloss.item()
                s["entropy"] += ent_b.item(); s["sigma"] += ent_b.item()
                s["approx_kl"] += mb_kl
                s["clipfrac"] += (torch.abs(ratio - 1.0) > CLIP).to(DTYPE).mean().item()
                s["grad_norm"] += float(gnorm)
                ep_kl.append(mb_kl)
            nsteps += 1
            if KL_STOP_MINIBATCH and mb_kl > KL_HARD_MULT * KL_TARGET:
                break
        if KL_STOP_MINIBATCH and ep_kl and ep_kl[-1] > KL_HARD_MULT * KL_TARGET:
            break
        if KL_EARLYSTOP and ep_kl and (sum(ep_kl) / len(ep_kl)) > KL_TARGET:   # end-of-epoch (mean), not mid-minibatch
            break
    if nsteps:
        for k in s:
            s[k] /= nsteps
    return s


## 13. Train loop (Elo-league driven, no hand-coded stages)

Iteration-driven curriculum, logstd-cap anneal (phase 1: force cap `LOGSTD_MAX -> LOGSTD_MAX_END`
over `SIGMA_DECAY_ITERS`; phase 2: release to `LOGSTD_MAX_POST`), stage-4 self-play snapshot
(refreshed every `SELFPLAY_REFRESH` iters). Per-iter heartbeat: stage, real return, launches/step,
approx-KL, clipfrac, sigma. Returns logged history for plotting.

**v8**: dual-Elo league (2p + 4p ratings, cross-coupled), invited members from the league folder (anchored, boost-then-drop), weak-point matchmaking (boost members the learner underperforms its Elo expectation against, up to x3), and the bounded error-diffusion 2p/4p scheduler (1:1 .. 1:6).


In [ ]:
import copy
import glob
import re

# ---------------- dual-Elo self-play league (PFSP pool + invited members) ------------------
def _freeze_snapshot(net):
    """Move to CPU, eval mode, no grad -> a fixed sparring partner."""
    net = net.cpu().eval()
    for p in net.parameters():
        p.requires_grad_(False)
    return net


class LegacyObsAdapter(torch.nn.Module):
    """Wraps a league member saved with the older SCALAR-ownership obs (3 fewer body features:
    v7/v8 F_DIM=101) so it can play inside a one-hot build (v9, F_DIM=104): the 4 seat channels
    collapse back to the scalar code the member was trained on (1 = self, 2 = ANY enemy,
    0 = neutral). DORMANT whenever the member's F_DIM matches the current build's."""
    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, ent, em, am, gl):
        b7 = ent[..., 7:8] + 2.0 * ent[..., 8:11].sum(-1, keepdim=True).clamp(max=1.0)
        return self.net(torch.cat([ent[..., :7], b7, ent[..., 11:]], -1), em, am, gl)


def _unwrap(net):
    return net.net if isinstance(net, LegacyObsAdapter) else net


def _wrap_if_legacy(net, cfg):
    """Current-format members pass through; v7-format (3 fewer body features) get the obs
    adapter; anything else is incompatible."""
    fd = int((cfg or {}).get("F_DIM", F_DIM))
    if fd == F_DIM:
        return net
    if fd == F_DIM - 3:
        return LegacyObsAdapter(net)
    raise ValueError("incompatible member F_DIM %d (current %d)" % (fd, F_DIM))


_POLICY_KW = ("HIDDEN", "D_G", "N_RES_BLOCKS", "USE_GLU", "USE_ATTENTION", "VALUE_RES_BLOCKS",
              "ARCH", "N_HEADS", "N_TX_LAYERS", "TX_MLP_RATIO", "N_STEM_RES", "N_HEAD_RES",
              "F_DIM")


def _cfg_from(blob_cfg, fallback=None):
    """Per-member PolicyNet dims: ckpt config > fallback (e.g. INVITE_CFG) > notebook globals."""
    out = {}
    for k in _POLICY_KW:
        if blob_cfg and k in blob_cfg:
            out[k] = blob_cfg[k]
        elif fallback and k in fallback:
            out[k] = fallback[k]
        else:
            out[k] = globals()[k]
    return out


def build_from_cfg(cfg):
    return PolicyNet(int(cfg.get("F_DIM", F_DIM)), G_DIM, cfg["HIDDEN"], cfg["D_G"], PLANET_CAP,
                     cfg["N_RES_BLOCKS"], cfg["USE_GLU"], INIT_GATE_BIAS,
                     use_attention=cfg["USE_ATTENTION"], value_res_blocks=cfg["VALUE_RES_BLOCKS"],
                     arch=cfg["ARCH"], n_heads=cfg["N_HEADS"], n_tx_layers=cfg["N_TX_LAYERS"],
                     tx_mlp_ratio=cfg["TX_MLP_RATIO"], n_stem_res=cfg["N_STEM_RES"],
                     n_head_res=cfg["N_HEAD_RES"])


def load_snapshot(path, fallback_cfg=None):
    """Load a save_ckpt() .pt into a frozen PolicyNet on CPU. Returns (net, cfg) -- the cfg is
    stored on the member so resumable states can rebuild differently-sized (invited) nets."""
    blob = torch.load(path, map_location="cpu", weights_only=False)
    cfg = _cfg_from(blob.get("config", {}), fallback_cfg)
    net = build_from_cfg(cfg)
    net.load_state_dict(blob["model"])
    return _freeze_snapshot(_wrap_if_legacy(net, cfg)), cfg


# ---------------- invited members: folder discovery + anchored Elo ------------------
_ELO_RE = re.compile(r"_elo(-?\d+)", re.IGNORECASE)


def _invited_elo(path):
    """Manually-anchored Elo of an invited ckpt: filename `_elo<NNNN>` > ckpt meta > default."""
    m = _ELO_RE.search(os.path.basename(path))
    if m:
        return float(m.group(1))
    try:
        blob = torch.load(path, map_location="cpu", weights_only=False)
        if blob.get("elo") is not None:
            return float(blob["elo"])
    except Exception:
        pass
    return float(INVITE_DEFAULT_ELO)


def discover_invited():
    """Scan the FIRST existing INVITE_DIRS folder for *.pt members; return the top INVITE_MAX
    [(path, anchored_elo)] sorted by Elo (desc)."""
    for d in INVITE_DIRS:
        if d and os.path.isdir(d):
            cands = sorted(glob.glob(os.path.join(d, "*.pt")))
            if cands:
                ranked = sorted(((p, _invited_elo(p)) for p in cands), key=lambda t: -t[1])
                print("[league] invite dir %s: %d candidate(s) -> inviting top %d"
                      % (d, len(ranked), min(INVITE_MAX, len(ranked))))
                return ranked[:INVITE_MAX]
    print("[league] no invite dir found (searched: %s)" % ", ".join(x for x in INVITE_DIRS if x))
    return []


class League:
    """Dual-Elo league. Every member carries a 2p rating (elo) AND a 4p rating (elo4); the
    learner keeps learner_elo / learner_elo4. Scripted anchors AND invited members are
    rating-ANCHORED in both formats (they pin the scales). Snapshots are PFSP-sampled with
    per-format ratings; invited members boost while unmastered, then their matchup rate DROPS
    to INVITE_MASTERED_W. share_4p() turns the (elo2 - elo4) gap into the bounded match mix."""
    def __init__(self, rng, invite=True):
        self.rng = rng
        self.learner_elo = ELO_INIT
        self.learner_elo4 = ELO_INIT
        self.advanced = False                                # curriculum: greedy/medium locked until the advance gate
        def anc(label, kind, elo):
            return {"label": label, "net": None, "kind": kind, "elo": elo, "elo4": elo,
                    "anchor": True, "pinned": True, "n": 0, "n4": 0}
        self.members = [
            anc("random", "random", ELO_RANDOM),
            anc("starter", "starter", ELO_STARTER),
            anc("greedy", "greedy", ELO_GREEDY),
            anc("medium", "medium", ELO_MEDIUM),
            anc("intermediate", "intermediate", ELO_INTERMEDIATE),
        ]
        if invite:
            for path, aelo in discover_invited():
                try:
                    self.add_invited(path, aelo)
                except Exception as e:
                    print("  !! invite failed %s -> %r" % (path, e))

    def add_invited(self, path, aelo):
        net, cfg = load_snapshot(path, fallback_cfg=INVITE_CFG)
        lab = os.path.splitext(os.path.basename(path))[0]
        m = {"label": "inv:" + lab, "net": net, "kind": "invited", "cfg": cfg,
             "elo": float(aelo), "elo4": float(aelo) + INVITE_ELO4_OFFSET,
             "anchor": True, "pinned": True, "n": 0, "n4": 0}
        self.members.append(m)
        print("    invited %-26s anchored elo2 %6.0f / elo4 %6.0f  (%s)"
              % (m["label"], m["elo"], m["elo4"], os.path.basename(path)))
        return m

    def snapshots(self):
        return [m for m in self.members if m["kind"] == "snapshot"]

    def anchor(self, kind):
        return next(m for m in self.members if m["kind"] == kind)

    def add_checkpoint(self, path, label=None, elo=None):
        net, cfg = load_snapshot(path)
        m = {"label": label or os.path.basename(path), "net": net, "kind": "snapshot", "cfg": cfg,
             "elo": ELO_INIT if elo is None else elo, "elo4": ELO_INIT if elo is None else elo,
             "anchor": False, "pinned": True, "n": 0, "n4": 0}
        self.members.append(m)
        return m

    def add_learner_snapshot(self, net, it):
        snap = _freeze_snapshot(copy.deepcopy(net))
        m = {"label": "it%d" % it, "net": snap, "kind": "snapshot",
             "elo": self.learner_elo, "elo4": self.learner_elo4,
             "anchor": False, "pinned": False, "n": 0, "n4": 0}
        self.members.append(m)
        self._prune()
        try:  # persist the weights as soon as the snapshot is generated (survives pruning)
            d = os.path.join(CKPT_DIR, "league_agents"); os.makedirs(d, exist_ok=True)
            save_ckpt(snap, os.path.join(d, m["label"] + ".pt"),
                      meta={"iter": it, "elo": float(m["elo"]), "elo4": float(m["elo4"])})
        except Exception as e:
            print("  !! league snapshot save failed: %r" % e)

    def _prune(self):
        autos = [m for m in self.members if m["kind"] == "snapshot" and not m["pinned"]]
        for m in autos[:max(0, len(autos) - LEAGUE_MAX_SNAPSHOTS)]:   # drop the oldest (FIFO)
            self.members.remove(m)

    # ---------------- dual-Elo machinery ----------------
    def _p_beat(self, m, fmt):
        me = self.learner_elo if fmt == 2 else self.learner_elo4
        oe = m["elo"] if fmt == 2 else m["elo4"]
        return 1.0 / (1.0 + 10.0 ** ((oe - me) / 400.0))

    def _pfsp_weight(self, p):
        if PFSP_MODE == "hard":
            return (1.0 - p) ** PFSP_POWER + PFSP_FLOOR   # focus on opponents you can't yet beat
        if PFSP_MODE == "even":
            return p * (1.0 - p) + PFSP_FLOOR             # focus on evenly-matched opponents
        return 1.0                                        # uniform

    def _mastered(self, m, fmt, thr):
        n = m["n"] if fmt == 2 else m.get("n4", 0)
        wr = m.get("wr") if fmt == 2 else m.get("wr4")
        return (n >= SCRIPT_BOOST_MIN_GAMES) and ((wr if wr is not None else 0.0) >= thr)

    def _weak_boost(self, m, fmt):
        """Weak-point matchmaking. If the learner's ACTUAL score EMA vs this member runs below
        the Elo-EXPECTED score (e.g. Elo says 0.90 but reality is 0.70), boost the matchup rate:
        1 + WEAK_BOOST_GAIN*(expected - actual), capped at WEAK_BOOST_MAX (x3 at the 0.90-vs-0.70
        example). ONLY targets weakness: overperforming opponents are never down-weighted, and
        the detector stays off below WEAK_BOOST_MIN_GAMES matches / inside the deadband."""
        if not WEAK_BOOST_ENABLED:
            return 1.0
        n = m["n"] if fmt == 2 else m.get("n4", 0)
        wr = m.get("wr") if fmt == 2 else m.get("wr4")
        if n < WEAK_BOOST_MIN_GAMES or wr is None:
            return 1.0
        gap = self._p_beat(m, fmt) - wr
        if gap < WEAK_BOOST_MIN_GAP:
            return 1.0
        return min(WEAK_BOOST_MAX, 1.0 + WEAK_BOOST_GAIN * gap)

    def _weights(self, fmt, net=None):
        """Per-member sampling weights for one format: PFSP(expected score) x footprint mix
        (2p only) x invited/scripted boosts x the advance lock."""
        members = self.members
        elo_w = [self._pfsp_weight(self._p_beat(m, fmt)) for m in members]
        ws = list(elo_w)
        if fmt == 2 and net is not None and MATCH_FP_W > 0.0:
            try:
                lfp = self._fingerprint(net)
                fd = []
                for m in members:
                    if m["net"] is None:
                        fd.append(1.0)                                  # scripted anchor: maximally distinct
                    else:
                        if m.get("fp") is None:
                            m["fp"] = self._fingerprint(m["net"])
                        fd.append(_js_div(lfp[0], m["fp"][0]) + _js_div(lfp[1], m["fp"][1]))
                mxe = max(elo_w) or 1.0; mxf = max(fd) or 1.0
                ws = [MATCH_ELO_W * (elo_w[i] / mxe) + MATCH_FP_W * (fd[i] / mxf) + PFSP_FLOOR
                      for i in range(len(members))]
            except Exception:
                ws = list(elo_w)
        for i, m in enumerate(members):
            if m["kind"] == "invited":
                # high-level sparring focus until mastered, then DROP the matchup rate
                ws[i] *= INVITE_MATCH_BOOST if not self._mastered(m, fmt, INVITE_MASTER_WR) else INVITE_MASTERED_W
            elif m["net"] is None and SCRIPT_MATCH_BOOST != 1.0:
                if not self._mastered(m, fmt, SCRIPT_BOOST_DROP_WR):
                    ws[i] *= SCRIPT_MATCH_BOOST              # spend compute on UNMASTERED scripted anchors
            ws[i] *= self._weak_boost(m, fmt)                # weak-point matchmaking (>= 1.0)
            if (not self.advanced) and m["kind"] in ("greedy", "medium"):
                ws[i] = 0.0                                  # curriculum lock until the advance gate
        return ws

    def sample(self, net=None, fmt=2):
        """PFSP-sample ONE opponent using the per-format ratings + boosts."""
        ws = self._weights(fmt, net)
        tot = sum(ws)
        if tot <= 0.0:
            return self.rng.choice(self.members)
        r = self.rng.random() * tot
        c = 0.0
        for m, w in zip(self.members, ws):
            c += w
            if r <= c:
                return m
        return self.members[-1]

    def sample_seats(self, net=None, k=3, fmt=4):
        """k seats for an FFA: weighted draws WITHOUT replacement while distinct members with
        positive weight remain (falls back to with-replacement on a tiny pool)."""
        ws = self._weights(fmt, net)
        out = []
        for _ in range(k):
            tot = sum(ws)
            if tot <= 0.0:
                out.append(self.rng.choice(self.members))
                continue
            r = self.rng.random() * tot
            c = 0.0; pick = len(self.members) - 1
            for i, w in enumerate(ws):
                c += w
                if r <= c:
                    pick = i
                    break
            out.append(self.members[pick])
            if sum(1 for w in ws if w > 0.0) > 1:
                ws[pick] = 0.0                                # without replacement while we can
        return out

    def update_elo(self, m, learner_score):
        """2p result: learner_score in [0,1] = win + 0.5*draw over the rollout. Anchors stay
        fixed. The 2p delta also drags BOTH 4p ratings by ELO_COUPLING (correlated ratings)."""
        exp = self._p_beat(m, 2)
        delta = ELO_K * (learner_score - exp)
        self.learner_elo += delta
        self.learner_elo4 += ELO_COUPLING * delta
        m["n"] += 1
        m["wr"] = learner_score if m.get("wr") is None else (1.0 - WR_EMA_BETA) * m["wr"] + WR_EMA_BETA * learner_score
        if not m["anchor"]:
            m["elo"] -= delta
            m["elo4"] -= ELO_COUPLING * delta

    def update_elo_4p(self, seats, seat_scores):
        """4p FFA result as len(seats) pairwise updates at ELO_K4/len each (standard multiplayer
        decomposition). Coupled back into the 2p ratings by ELO_COUPLING."""
        kf = ELO_K4 / max(1, len(seats))
        for m, sc in zip(seats, seat_scores):
            exp = self._p_beat(m, 4)
            delta = kf * (sc - exp)
            self.learner_elo4 += delta
            self.learner_elo += ELO_COUPLING * delta
            m["n4"] = m.get("n4", 0) + 1
            m["wr4"] = sc if m.get("wr4") is None else (1.0 - WR_EMA_BETA) * m["wr4"] + WR_EMA_BETA * sc
            if not m["anchor"]:
                m["elo4"] -= delta
                m["elo"] -= ELO_COUPLING * delta

    def share_4p(self):
        """Share of 4p iterations from the rating gap, clipped to the 1:1 .. 1:6 envelope.
        4p lagging (elo2 > elo4) -> more 4p; 4p ahead -> drift back toward 2p."""
        gap = (self.learner_elo - self.learner_elo4) / 400.0
        return min(S4_MAX, max(S4_MIN, S4_BASE + S4_GAIN * gap))

    def leaderboard(self):
        rows = sorted(self.members, key=lambda m: -m["elo"])
        s = "  league | learner elo2 %.0f elo4 %.0f | share4p %.2f | %d members\n" % (
            self.learner_elo, self.learner_elo4, self.share_4p(), len(self.members))
        for m in rows:
            tag = "[anchor]" if m["anchor"] else ("[pinned]" if m["pinned"] else "")
            if m["kind"] == "invited":
                tag = "[invited %s%s]" % ("M" if self._mastered(m, 2, INVITE_MASTER_WR) else "-",
                                          "M" if self._mastered(m, 4, INVITE_MASTER_WR) else "-")
            elif m["net"] is None:
                boosted = not self._mastered(m, 2, SCRIPT_BOOST_DROP_WR)
                tag += " x%.1f" % SCRIPT_MATCH_BOOST if boosted else " (mastered)"
            w2, w4 = self._weak_boost(m, 2), self._weak_boost(m, 4)
            if w2 > 1.0:
                tag += " weak2 x%.1f" % w2
            if w4 > 1.0:
                tag += " weak4 x%.1f" % w4
            wr = m.get("wr"); wr4 = m.get("wr4")
            s += "    %-22s elo2 %6.0f (n=%-3d wr %s)  elo4 %6.0f (n4=%-3d wr %s) %s\n" % (
                m["label"], m["elo"], m["n"], ("%.2f" % wr) if wr is not None else " -- ",
                m["elo4"], m.get("n4", 0), ("%.2f" % wr4) if wr4 is not None else " -- ", tag)
        return s


def save_ckpt(net, path, meta=None):
    blob = {"model": net.state_dict(),
            "config": {"HIDDEN": HIDDEN, "N_RES_BLOCKS": N_RES_BLOCKS, "D_G": D_G,
                       "PLANET_CAP": PLANET_CAP, "F_DIM": F_DIM, "G_DIM": G_DIM,
                       "ARCH": ARCH, "N_TX_LAYERS": N_TX_LAYERS, "N_HEADS": N_HEADS,
                       "TX_MLP_RATIO": TX_MLP_RATIO, "N_STEM_RES": N_STEM_RES, "N_HEAD_RES": N_HEAD_RES,
                       "USE_GLU": USE_GLU, "USE_ATTENTION": USE_ATTENTION, "VALUE_RES_BLOCKS": VALUE_RES_BLOCKS,
                       "target_actor": True}}
    if meta:
        blob.update(meta)
    # league snapshots can be SMALLER than the live config (invited dims): persist the true dims
    if hasattr(net, "trunk") and hasattr(net.trunk, "layers"):
        blob["config"]["HIDDEN"] = net.h
        blob["config"]["N_TX_LAYERS"] = len(net.trunk.layers)
        blob["config"]["N_HEADS"] = net.trunk.layers[0].nh if net.trunk.layers else N_HEADS
        blob["config"]["N_STEM_RES"] = len(net.trunk.stem_a)
        blob["config"]["N_HEAD_RES"] = len(net.trunk.head_a)
    torch.save(blob, path)


def anneal_ent_coef(it):
    if ENT_DECAY_ITERS <= 0:
        return ENT_COEF_END
    f = min(1.0, it / ENT_DECAY_ITERS)
    return ENT_COEF_START + (ENT_COEF_END - ENT_COEF_START) * f


def train(total_iters=TOTAL_ITERS, log_every=1, resume_from=None):
    net = build_policy()
    try:
        opt = torch.optim.Adam(net.parameters(), lr=LR, eps=ADAM_EPS, fused=(DEVICE.type == 'cuda'))
    except Exception:
        opt = torch.optim.Adam(net.parameters(), lr=LR, eps=ADAM_EPS)
    env = GpuEnv(PLANET_CAP, FLEET_CAP, EPISODE_STEPS, SHIP_SPEED, DEVICE)
    rng = random.Random(SEED * 2654435761 + 12345)
    start_it = 0
    best_wr = -1.0

    league = League(rng)
    for ck in LEAGUE_INIT_CKPTS:
        try:
            m = league.add_checkpoint(ck)
            print("league seeded with %s  <- %s" % (m["label"], ck))
        except Exception as e:
            print("  !! skipped league ckpt %s -> %r" % (ck, e))

    if resume_from:
        start_it, best_wr = load_train_state(net, opt, league, resume_from)
        print("resumed from %s -> global iter %d, best_wr %.2f, %d league members"
              % (resume_from, start_it, best_wr, len(league.members)))

    hist = {"iter": [], "return": [], "win_rate": [],
            "lnch_per_step": [], "approx_kl": [], "clipfrac": [], "sigma": [], "grad_norm": [],
            "r_outcome": [], "r_capture": [], "r_milestone": [], "r_launch": [],
            "learner_elo": [], "learner_elo4": [], "share_4p": [], "fmt": []}

    pool, pool_round = None, -1
    # held-out gauntlet worlds: FIXED seed range never touched by training rounds, so the
    # best-ckpt score measures generalization and stays comparable across the whole run.
    heldout_pool = make_world_pool(ELO_RECAL_ENVS, base_seed=SEED + 999_999_937)
    acc4 = 0.0       # error-diffusion accumulator -> the realized 2p:4p ratio equals share_4p
    for step in range(total_iters):
        it = start_it + step
        rnd = (it // WORLD_RESAMPLE_EVERY) if WORLD_RESAMPLE_EVERY > 0 else 0
        if rnd != pool_round:
            pool = make_world_pool(N_WORLDS, base_seed=SEED + rnd * N_WORLDS)
            cursor, pool_round = 0, rnd
            print("  [worlds] pool round %d @ it%d (base_seed %d)" % (rnd, it, SEED + rnd * N_WORLDS))
        if ELO_RECAL_EVERY > 0 and step > 0 and it % ELO_RECAL_EVERY == 0:  # re-ground BOTH rating scales
            league.recalibrate_elo(net, pool)
            if FOURP_ENABLED:
                league.recalibrate_elo4(net, pool)
        t0 = time.time()
        globals()['CUR_ENT_COEF'] = anneal_ent_coef(it)   # entropy anneal (anti collapse -> exploit)
        # v7 guardrails: linear LR warmup + critic-only warmup (both per train() CALL, not global iter)
        globals()['PPO_POLICY_COEF'] = 0.0 if step < VALUE_WARMUP_ITERS else 1.0
        if LR_WARMUP_ITERS > 0:
            wf = min(1.0, (step + 1) / float(LR_WARMUP_ITERS))
            for pg in opt.param_groups:
                pg['lr'] = LR * wf

        # grow the pool: snapshot the learner (after a short anchors-only warmup) every SELFPLAY_REFRESH iters
        if (it >= SNAPSHOT_WARMUP) and (it % max(1, SELFPLAY_REFRESH) == 0):
            league.add_learner_snapshot(net, it + 1)

        # ---- v8 match-type scheduler: dual-Elo gap -> bounded 2p/4p mix (1:1 .. 1:6) ----
        s4 = league.share_4p() if FOURP_ENABLED else 0.0
        acc4 += s4
        if acc4 >= 1.0:
            fmt = 4; acc4 -= 1.0
        else:
            fmt = 2

        # pick the opponent seat(s) for this rollout: PFSP over the whole league per format
        if fmt == 2:
            seats = [league.sample(net, fmt=2)]
        else:
            seats = league.sample_seats(net, k=3, fmt=4)
        for m in seats:
            if m["net"] is not None:
                m["net"].to(DEVICE)

        tb, rs, cursor = collect_ppo(env, net, pool, cursor, rng, seats,
                                     n_players=fmt, n_envs=(B if fmt == 2 else B_4P))
        us = ppo_update(net, opt, tb)

        # dual Elo: 2p = single update; 4p = pairwise vs every seat (cross-coupled inside)
        if fmt == 2:
            league.update_elo(seats[0], rs["score"])
        else:
            league.update_elo_4p(seats, rs["seat_scores"])
        if (not league.advanced) and league.learner_elo >= ADVANCE_TRIGGER_ELO:   # curriculum: confirm with an all-anchor recal (fresh seeds)
            league.recalibrate_elo(net, make_world_pool(ELO_RECAL_ENVS, base_seed=SEED + 104729 + 7919 * (it + 1)), all_anchors=True)
            if league.learner_elo >= ADVANCE_CONFIRM_ELO:
                league.advanced = True
                print("  [advance] learner hit %.0f ELO; all-anchor recal %.0f >= %d -> UNLOCK greedy/medium" % (ADVANCE_TRIGGER_ELO, league.learner_elo, ADVANCE_CONFIRM_ELO))
            else:
                print("  [advance] learner hit %.0f ELO but all-anchor recal %.0f < %d -> stay in base regime" % (ADVANCE_TRIGGER_ELO, league.learner_elo, ADVANCE_CONFIRM_ELO))
        for m in seats:
            if m["net"] is not None:
                m["net"].to("cpu")           # keep only the live policy resident on the GPU

        dt = time.time() - t0
        sps = rs["transitions"] / max(dt, 1e-9)

        hist["iter"].append(it + 1)
        hist["return"].append(rs["mean_return"]); hist["win_rate"].append(rs["win_rate"])
        hist["lnch_per_step"].append(rs["lnch_per_step"]); hist["approx_kl"].append(us["approx_kl"])
        hist["clipfrac"].append(us["clipfrac"]); hist["sigma"].append(us["sigma"]); hist["grad_norm"].append(us["grad_norm"])
        hist["r_outcome"].append(rs["r_outcome"]); hist["r_capture"].append(rs["r_capture"])
        hist["r_milestone"].append(rs["r_milestone"]); hist["r_launch"].append(rs["r_launch"])
        hist["learner_elo"].append(league.learner_elo)
        hist["learner_elo4"].append(league.learner_elo4)
        hist["share_4p"].append(s4); hist["fmt"].append(fmt)

        # v7 tripwire: gate-collapse detector (launch rate decaying under healthy-looking KL)
        _l = hist["lnch_per_step"]
        if len(_l) >= TRIPWIRE_BASE_ITERS and (it + 1) % 10 == 0:
            _base = sum(_l[:TRIPWIRE_BASE_ITERS]) / TRIPWIRE_BASE_ITERS
            _ema = sum(_l[-5:]) / len(_l[-5:])
            if _base > 0 and _ema < TRIPWIRE_FRAC * _base:
                print("  [TRIPWIRE] launch/st EMA %.2f < %.0f%% of early baseline %.2f -> passivity ratchet suspected; check gauntlet+elo before continuing" % (_ema, TRIPWIRE_FRAC * 100, _base))

        if (it + 1) % log_every == 0 or step == 0 or step == total_iters - 1:
            # heartbeat: R[o c p ln] = outcome, capture, prod-milestone, launch (real units)
            opp_lab = "+".join(m["label"] for m in seats)
            print("it%4d %dp | ret %8.2f wr %.2f sc %.2f | R[o %7.1f c %6.1f p %5.1f ln %5.1f] | "
                  "lnch/st %.2f | kl %.3f cf %.2f gn %.1f | loss %7.3f (pol %.3f vf %.3f) | "
                  "elo2 %5.0f elo4 %5.0f s4 %.2f vs %-22s | sps %5.0f"
                  % (it + 1, fmt, rs["mean_return"], rs["win_rate"], rs["score"],
                     rs["r_outcome"], rs["r_capture"], rs["r_milestone"], rs["r_launch"],
                     rs["lnch_per_step"], us["approx_kl"], us["clipfrac"], us["grad_norm"],
                     us["total"], us["policy"], us["vf"],
                     league.learner_elo, league.learner_elo4, s4, opp_lab[:22], sps))

        # checkpoint: periodic + best-by-MIXED-GAUNTLET (2p + 4p; fixed opponents, held-out worlds)
        if ((it + 1) % max(1, CKPT_EVERY) == 0) or (step == total_iters - 1):
            save_ckpt(net, CKPT_PATH, meta={"iter": it + 1, "win_rate": rs["win_rate"]})
            save_train_state(net, opt, league, it + 1, best_wr, TRAIN_STATE_PATH)
            g2 = eval_gauntlet(net, heldout_pool)
            g4 = eval_gauntlet4(net, heldout_pool) if FOURP_ENABLED else g2
            gscore = (1.0 - GAUNTLET_4P_W) * g2 + GAUNTLET_4P_W * g4
            hist.setdefault("gauntlet", []).append(gscore)
            hist.setdefault("gauntlet2", []).append(g2)
            hist.setdefault("gauntlet4", []).append(g4)
            if gscore > best_wr:
                best_wr = gscore
                save_ckpt(net, BEST_CKPT_PATH, meta={"iter": it + 1, "gauntlet": float(gscore),
                                                     "gauntlet2": float(g2), "gauntlet4": float(g4)})
                print("  [best] mixed gauntlet %.3f (2p %.3f / 4p %.3f) -> saved best" % (gscore, g2, g4))
            print(league.leaderboard(), end="")

    save_ckpt(net, CKPT_PATH, meta={"iter": start_it + total_iters, "win_rate": hist["win_rate"][-1]})
    save_train_state(net, opt, league, start_it + total_iters, best_wr, TRAIN_STATE_PATH)
    print("saved final checkpoint ->", CKPT_PATH, "| best mixed gauntlet %.2f ->" % best_wr, BEST_CKPT_PATH)
    print(league.leaderboard(), end="")
    import json as _json
    _json.dump({k: [float(x) for x in v] for k, v in hist.items()},
               open(os.path.join(CKPT_DIR, "metrics.json"), "w"))
    print("saved metrics ->", os.path.join(CKPT_DIR, "metrics.json"))
    globals()["LEAGUE"] = league      # also expose as a module global (robust if the caller drops it)
    return net, hist, league


## 14. Run training

This is the heavy cell. With SMOKE=True it is a short sanity run; flip SMOKE off (section 2)
and re-run for the full spec. On a Colab/Kaggle T4 the full run takes hours -- reduce B,
HIDDEN, N_RES_BLOCKS, or TOTAL_ITERS to fit your budget. Checkpoints are written each cadence.

In [ ]:
# ---- resumable training state + fixed-opponent gauntlets (2p AND 4p) + Elo re-grounding ----
import math

def eval_gauntlet(net, worlds, anchors=("starter", "intermediate", "greedy"), n_envs=None):
    """Fixed-opponent 2p score for best-ckpt selection (greedy, deployment-faithful op set)."""
    n = n_envs or ELO_RECAL_ENVS
    return sum(_eval_score(net, a, worlds, n) for a in anchors) / len(anchors)


GAUNTLET4_SEATS = ("starter", "greedy", "intermediate")   # fixed FFA trio for the 4p gauntlet

_OPP_BY_KIND = {"random": 0, "starter": 1, "noop": 2, "medium": 3, "greedy": 4, "intermediate": 5}


def _eval_score(net, anchor_kind, worlds, n_envs):
    """No-grad mean learner-score (win + 0.5*draw) of `net` vs a scripted anchor on `worlds` (greedy, 2p)."""
    env = GpuEnv(PLANET_CAP, FLEET_CAP, EPISODE_STEPS, SHIP_SPEED, DEVICE)
    env.reset([worlds[i % len(worlds)] for i in range(n_envs)])
    active = torch.ones(n_envs, device=DEVICE); score = torch.zeros(n_envs, device=DEVICE)
    def _sc(s0, s1):
        return torch.where(s0 > s1, torch.ones_like(s0), torch.where(s0 < s1, torch.zeros_like(s0), torch.full_like(s0, 0.5)))
    with torch.no_grad():
        for _t in range(EPISODE_STEPS):
            ent, em, am, gl = env_encode(env, 0)
            a_t, _, _ = act(net, ent, em, am, gl, greedy=True)
            env_step(env, a_t, _OPP_BY_KIND[anchor_kind], None, step_idx=_t)
            s0, s1, a0, a1 = settle(env)
            term = (env.step_ct >= float(env.T - 2)) | (~(a0 & a1))
            newly = (active > 0.5) & term
            score = torch.where(newly, _sc(s0, s1), score)
            active = torch.where(term, torch.zeros_like(active), active)
            if active.sum().item() == 0:
                break
        s0, s1, _, _ = settle(env)
        score = torch.where(active > 0.5, _sc(s0, s1), score)
    return float(score.mean().item())


def _eval_score4(net, trio, worlds, n_envs):
    """No-grad mean PAIRWISE score of greedy `net` (seat 0) vs 3 scripted anchors in a 4p FFA."""
    env = GpuEnv(PLANET_CAP, FLEET_CAP, EPISODE_STEPS, SHIP_SPEED, DEVICE)
    env.reset([worlds[i % len(worlds)] for i in range(n_envs)], n_players=4)
    seats = [{"pid": p + 1, "script": _OPP_BY_KIND[k]} for p, k in enumerate(trio)]
    active = torch.ones(n_envs, device=DEVICE); score = torch.zeros(n_envs, device=DEVICE)
    def _pw(s_all):
        s0 = s_all[:, 0:1].expand_as(s_all[:, 1:]); rest = s_all[:, 1:]
        one = torch.ones_like(rest)
        return torch.where(s0 > rest, one, torch.where(s0 < rest, 0.0 * one, 0.5 * one)).mean(1)
    with torch.no_grad():
        for _t in range(EPISODE_STEPS):
            ent, em, am, gl = env_encode(env, 0)
            a_t, _, _ = act(net, ent, em, am, gl, greedy=True)
            env_step(env, a_t, seats=seats, step_idx=_t)
            s_all, alive_all = settle_n(env)
            n_alive = alive_all.to(DTYPE).sum(1)
            term = (env.step_ct >= float(env.T - 2)) | (n_alive <= 1.0) | (~alive_all[:, 0])
            newly = (active > 0.5) & term
            score = torch.where(newly, _pw(s_all), score)
            active = torch.where(term, torch.zeros_like(active), active)
            if active.sum().item() == 0:
                break
        s_all, _ = settle_n(env)
        score = torch.where(active > 0.5, _pw(s_all), score)
    return float(score.mean().item())


def eval_gauntlet4(net, worlds, n_envs=None):
    """Fixed-trio 4p FFA gauntlet (mean pairwise score vs starter/greedy/intermediate)."""
    return _eval_score4(net, GAUNTLET4_SEATS, worlds, n_envs or ELO_RECAL_ENVS)


# ---- league (de)serialization: dual ratings + per-member dims (invited nets are smaller) ----
def _league_state_dict(self):
    members = []
    for m in self.members:
        members.append({"label": m["label"], "kind": m["kind"], "elo": m["elo"],
                        "elo4": m.get("elo4", m["elo"]), "anchor": m["anchor"], "pinned": m["pinned"],
                        "n": m["n"], "n4": m.get("n4", 0), "wr": m.get("wr"), "wr4": m.get("wr4"),
                        "cfg": m.get("cfg"),
                        "model": (_unwrap(m["net"]).state_dict() if m["net"] is not None else None)})
    return {"learner_elo": self.learner_elo, "learner_elo4": self.learner_elo4,
            "advanced": self.advanced, "members": members}

def _league_load_state_dict(self, sd):
    self.learner_elo = sd["learner_elo"]
    self.learner_elo4 = sd.get("learner_elo4", sd["learner_elo"])
    self.advanced = sd.get("advanced", self.advanced)
    self.members = []
    for e in sd["members"]:
        net = None
        if e["model"] is not None:
            net = build_from_cfg(e["cfg"]) if e.get("cfg") else build_policy()
            net.load_state_dict(e["model"])
            net = _freeze_snapshot(_wrap_if_legacy(net, e.get("cfg")))
        m = {"label": e["label"], "kind": e["kind"], "net": net, "elo": e["elo"],
             "elo4": e.get("elo4", e["elo"]), "anchor": e["anchor"], "pinned": e["pinned"],
             "n": e["n"], "n4": e.get("n4", 0)}
        if e.get("cfg") is not None:
            m["cfg"] = e["cfg"]
        if e.get("wr") is not None:
            m["wr"] = e["wr"]
        if e.get("wr4") is not None:
            m["wr4"] = e["wr4"]
        self.members.append(m)

League.state_dict = _league_state_dict
League.load_state_dict = _league_load_state_dict


def save_train_state(net, opt, league, global_iter, best_wr, path):
    """Full resumable state: weights + Adam moments + global iter + best score + dual-Elo league."""
    pa = getattr(net, "popart", None)
    torch.save({"model": net.state_dict(), "opt": opt.state_dict(), "league": league.state_dict(),
                "global_iter": int(global_iter), "best_wr": float(best_wr),
                "popart": ({"mu": pa.mu, "nu": pa.nu, "sigma": pa.sigma, "init": pa.init} if pa else None)}, path)


def load_train_state(net, opt, league, path):
    """Restore net/opt/league IN PLACE. Returns (start_iter, best_wr). Accepts a full train-state
    OR a plain save_ckpt weight (warm-start: weights only, fresh Adam/league). v7 states load too
    (missing elo4 fields default to the 2p values); invited members are re-discovered if absent."""
    blob = torch.load(path, map_location=DEVICE, weights_only=False)
    net.load_state_dict(blob["model"])
    if blob.get("popart") and getattr(net, "popart", None) is not None:
        d = blob["popart"]; net.popart.mu = d["mu"]; net.popart.nu = d["nu"]; net.popart.sigma = d["sigma"]; net.popart.init = d["init"]
    if "opt" in blob:
        opt.load_state_dict(blob["opt"])
    else:
        print("  [resume] %s has no optimizer state -> WARM-START (fresh Adam moments)" % os.path.basename(path))
    start_it = int(blob.get("global_iter", blob.get("iter", 0)))
    if "league" in blob:
        league.load_state_dict(blob["league"])
        if not any(m["kind"] == "invited" for m in league.members):   # v7 state: re-invite
            for p, aelo in discover_invited():
                try:
                    league.add_invited(p, aelo)
                except Exception as e:
                    print("  !! invite failed %s -> %r" % (p, e))
    else:  # warm-start: calibrate learner Elo vs anchors so the first snapshot is not stamped Elo 0
        rnd = (start_it // WORLD_RESAMPLE_EVERY) if WORLD_RESAMPLE_EVERY > 0 else 0
        print("  [resume] no league state -> calibrating learner Elo vs anchors")
        league.recalibrate_elo(net, make_world_pool(ELO_RECAL_ENVS, base_seed=SEED + rnd * N_WORLDS))
        league.learner_elo4 = league.learner_elo
    return start_it, float(blob.get("best_wr", -1.0))


def _league_recalibrate_elo(self, net, worlds, n_envs=None, all_anchors=False):
    """Re-ground every 2p rating from win-rate vs the FIXED anchors on `worlds` (drift removal)."""
    n_envs = n_envs or ELO_RECAL_ENVS
    _ADV = ("greedy", "medium")
    anchors = [m for m in self.members if m["anchor"] and m["net"] is None
               and (all_anchors or self.advanced or m["kind"] not in _ADV)]
    def est(scores):  # scores: [(anchor_elo, learner_score), ...] -> averaged ELO estimate
        vals = []
        for ae, sc in scores:
            sc = min(max(sc, 0.02), 0.98)
            vals.append(ae + 400.0 * math.log10(sc / (1.0 - sc)))
        return sum(vals) / len(vals)
    self.learner_elo = est([(a["elo"], _eval_score(net, a["kind"], worlds, n_envs)) for a in anchors])
    for m in self.members:
        if m["anchor"] or m["net"] is None:
            continue
        m["net"].to(DEVICE)
        m["elo"] = est([(a["elo"], _eval_score(m["net"], a["kind"], worlds, n_envs)) for a in anchors])
        m["net"].to("cpu"); m["n"] = 0
    print("    [elo recal] learner=%.0f | %s" % (self.learner_elo,
          " ".join("%s=%.0f" % (m["label"], m["elo"]) for m in self.members if not m["anchor"])))

League.recalibrate_elo = _league_recalibrate_elo


def _league_recalibrate_elo4(self, net, worlds, n_envs=None):
    """Re-ground the 4p ratings: learner + every snapshot eval'd in 4p FFA vs the fixed scripted
    trio; rating = mean trio Elo4 + the pairwise-score logit vs the field. Invited/scripted
    members stay anchored."""
    n_envs = n_envs or ELO_RECAL_ENVS
    base = sum(self.anchor(k)["elo4"] for k in GAUNTLET4_SEATS) / len(GAUNTLET4_SEATS)
    def est(s):
        s = min(max(s, 0.02), 0.98)
        return base + 400.0 * math.log10(s / (1.0 - s))
    self.learner_elo4 = est(_eval_score4(net, GAUNTLET4_SEATS, worlds, n_envs))
    for m in self.members:
        if m["anchor"] or m["net"] is None:
            continue
        m["net"].to(DEVICE)
        m["elo4"] = est(_eval_score4(m["net"], GAUNTLET4_SEATS, worlds, n_envs))
        m["net"].to("cpu"); m["n4"] = 0
    print("    [elo4 recal] learner4=%.0f | %s" % (self.learner_elo4,
          " ".join("%s=%.0f" % (m["label"], m["elo4"]) for m in self.members if not m["anchor"])))

League.recalibrate_elo4 = _league_recalibrate_elo4


# ---- league pruning: keep the most behaviorally DISTINCT snapshots (replaces FIFO) ----------
def _league_probe_obs(self):
    """Fingerprint probe = REAL encoded game states (on-manifold), cached once."""
    if getattr(self, '_probe', None) is None:
        env = GpuEnv(PLANET_CAP, FLEET_CAP, EPISODE_STEPS, SHIP_SPEED, DEVICE)
        env.reset(make_world_pool(16, base_seed=0xC0FFEE % 100000))
        with torch.no_grad():
            for _t in range(20):
                e, m, a, g = env_encode(env, 0)
                rnd = torch.softmax(torch.randn(env.B, PLANET_CAP, PLANET_CAP + 1, device=DEVICE), -1)
                env_step(env, rnd, 1, None, step_idx=_t)
        e, m, a, g = env_encode(env, 0)
        self._probe = (e.cpu(), m.cpu(), a.cpu(), g.cpu())
    return self._probe

def _league_fingerprint(self, net):
    """Behavioral fingerprint = mean (WHERE allocation, gate fire/hold) on the fixed probe batch."""
    ent, em, am, gl = self._probe_obs()
    dev = next(net.parameters()).device
    with torch.no_grad():
        dest_logits, gate_logits, _ = net(ent.to(dev), em.to(dev), am.to(dev), gl.to(dev))
        dest = torch.softmax(dest_logits, -1).reshape(-1, dest_logits.shape[-1]).mean(0)  # (E,)
        gp = torch.sigmoid(gate_logits).reshape(-1).float().mean()                         # mean fire-prob
        gate = torch.stack([gp, 1.0 - gp])                                                 # (2,) fire/hold
    return dest.cpu(), gate.cpu()

def _js_div(p, q):
    m = 0.5 * (p + q); eps = 1e-9
    kl = lambda a, b: (a * (torch.log(a + eps) - torch.log(b + eps))).sum()
    return float(0.5 * kl(p, m) + 0.5 * kl(q, m))

def _league_prune(self):
    """Keep LEAGUE_MAX_SNAPSHOTS most distinct auto-snapshots: repeatedly drop the one whose
    NEAREST neighbor (dest+gate Jensen-Shannon divergence) is closest = most redundant.
    Anchors / pinned ckpts and the NEWEST snapshot are always kept. FIFO fallback on error."""
    autos = [m for m in self.members if m['kind'] == 'snapshot' and not m['pinned']]
    if len(autos) <= LEAGUE_MAX_SNAPSHOTS:
        return
    try:
        fps = [self._fingerprint(m['net']) for m in autos]
    except Exception as e:
        print('  !! league diversity-prune -> FIFO fallback: %r' % e)
        for m in autos[:len(autos) - LEAGUE_MAX_SNAPSHOTS]:
            self.members.remove(m)
        return
    def d(i, j):
        return _js_div(fps[i][0], fps[j][0]) + _js_div(fps[i][1], fps[j][1])
    newest = len(autos) - 1                      # most recently appended -> always keep
    keep = list(range(len(autos)))
    while len(keep) > LEAGUE_MAX_SNAPSHOTS:
        drop, drop_nn = None, float('inf')
        for i in keep:
            if i == newest:
                continue
            nn = min(d(i, j) for j in keep if j != i)
            if nn < drop_nn:
                drop_nn, drop = nn, i
        if drop is None:
            break
        keep.remove(drop)
    kept = set(keep)
    for idx, m in enumerate(autos):
        if idx not in kept:
            self.members.remove(m)
    print('  league diversity-prune: kept %d/%d snapshots (max behavioral distinctness)'
          % (len(kept), len(autos)))

League._probe_obs = _league_probe_obs
League._fingerprint = _league_fingerprint
League._prune = _league_prune


In [ ]:
# ## 13b. BC warm-start -- clone the MEDIUM bot into the gated-alloc heads (v7)
#
# The from-scratch PPO bootstrap is the proven failure mode (passivity trap / giant first
# updates). BC hands PPO a committing, aiming policy from step 1:
#   expert  = the medium scripted rule (nearest non-owned planet, lead-aim), the strongest
#             scripted anchor, expressed DIRECTLY in the gated-alloc action space:
#             fire* = owned & alive & floor(ships/2) >= 20 & has-target;  where* = one-hot(nearest)
#   states  = visited by the CLONE itself (full-garrison medium) vs mixed scripted opponents
#             -> no train/deploy distribution gap on the states BC actually fits.
#   loss    = BCE(gate fire-prob, fire*) + CE(masked WHERE logits, nearest-target) on firing
#             sources only; sun-blocked expert targets are dropped (cloning "medium minus
#             sun-suicide"). The CE peaks softplus-alpha on the target, so the Dirichlet
#             concentrates exactly where the greedy sharpened-softmax looks.

def medium_targets_ego(env):
    """The medium rule for EGO (owner 0): (fire*, tgt) both (B,Ec)."""
    half = torch.floor(env.p_ships / 2.0)
    base = (env.p_owner == 0.0) & (env.p_alive > 0.5) & (half >= 20.0)
    sx = env.p_x.unsqueeze(2); sy = env.p_y.unsqueeze(2)
    tx = env.p_x.unsqueeze(1); ty = env.p_y.unsqueeze(1)
    vt = (env.p_alive > 0.5) & (env.p_owner != 0.0)
    d = torch.sqrt((sx - tx) ** 2 + (sy - ty) ** 2)
    dmask = torch.where(vt.unsqueeze(1), d, torch.full_like(d, BIG))
    bestd, tgt = dmask.min(2)
    fire = base & (bestd < BIG * 0.5)
    return fire, tgt


def bc_expert_action(env):
    """(B,Ec,Ec+1) gated-alloc action bundle of the expert: one-hot WHERE rows + fire."""
    fire, tgt = medium_targets_ego(env)
    B_, Ec = fire.shape
    alloc = torch.zeros(B_, Ec, Ec, dtype=DTYPE, device=env.dev)
    alloc.scatter_(2, tgt.unsqueeze(-1), 1.0)
    return torch.cat([alloc, fire.to(DTYPE).unsqueeze(-1)], -1), fire, tgt


def bc_pretrain(net=None, rounds=BC_ROUNDS, epochs=BC_EPOCHS, lr=BC_LR, log=True):
    """Supervised clone of the medium rule. Returns the net; saves BC_CKPT_PATH."""
    net = net or build_policy()
    opt = torch.optim.Adam(net.parameters(), lr=lr, eps=ADAM_EPS)
    env = GpuEnv(PLANET_CAP, FLEET_CAP, EPISODE_STEPS, SHIP_SPEED, DEVICE)
    pool = make_world_pool(max(B, 256), base_seed=SEED + 900001)
    opps = [1, 0, 3, 1, 4, 1, 3, 0, 1, 3, 4, 1]          # starter-heavy mix, incl. medium/greedy
    rng = random.Random(SEED + 31337)
    for r in range(rounds):
        worlds = [pool[rng.randrange(len(pool))] for _ in range(B)]
        env.reset(worlds)
        opp = opps[r % len(opps)]
        ents, ems, ams, gls, fires, tgts, actives = [], [], [], [], [], [], []
        active = torch.ones(B, device=DEVICE)
        with torch.no_grad():
            for t in range(env.T):
                ent, em, am, gl = env_encode(env, 0)
                a_t, fire, tgt = bc_expert_action(env)
                # stash on CPU: a full on-GPU rollout of encoded states is ~0.6 GB
                ents.append(ent.cpu()); ems.append(em.cpu()); ams.append(am.cpu()); gls.append(gl.cpu())
                fires.append(fire.cpu()); tgts.append(tgt.cpu()); actives.append(active.cpu().clone())
                env_step(env, a_t, opp, None, step_idx=t)
                s0, s1, side0, side1 = settle(env)
                active = active * (side0 & side1).to(DTYPE)
        ent = torch.cat(ents); em = torch.cat(ems); am = torch.cat(ams); gl = torch.cat(gls)
        fire = torch.cat(fires); tgt = torch.cat(tgts); act_m = torch.cat(actives)
        keep = act_m > 0.5                                    # drop post-terminal states
        ent, em, am, gl, fire, tgt = ent[keep], em[keep], am[keep], gl[keep], fire[keep], tgt[keep]
        N = ent.shape[0]
        mbs = max(8, BC_MINIBATCH // PLANET_CAP)              # boards per minibatch (~85 @ 4096/48)
        stats = [0.0, 0.0, 0.0, 0]
        for ep in range(epochs):
            perm = torch.randperm(N)
            for b in range(0, N, mbs):
                mi = perm[b:b + mbs]
                ent_d, em_d, am_d, gl_d = (x[mi].to(DEVICE) for x in (ent, em, am, gl))
                fire_d, tgt_d = fire[mi].to(DEVICE), tgt[mi].to(DEVICE)
                dist, _ = _make_dist(net, ent_d, em_d, am_d, gl_d)
                own = am_d > 0.5
                f = fire_d.to(DTYPE)
                p = dist.p                                     # smooth fire-prob (post-fix)
                bce = -(f * torch.log(p.clamp_min(1e-8)) + (1 - f) * torch.log((1 - p).clamp_min(1e-8)))
                bce = (bce * own.to(DTYPE)).sum() / own.sum().clamp_min(1)
                lsm = torch.log_softmax(dist.glogits, -1)      # masked WHERE logits (alive+reach+no-self)
                ce_all = -lsm.gather(2, tgt_d.unsqueeze(-1)).squeeze(-1)
                ok = own & fire_d & (dist.glogits.gather(2, tgt_d.unsqueeze(-1)).squeeze(-1) > -1e8)
                ce = (ce_all * ok.to(DTYPE)).sum() / ok.sum().clamp_min(1)
                loss = bce + ce
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), MAX_GRAD_NORM)
                opt.step()
                with torch.no_grad():
                    hit = ((lsm.argmax(-1) == tgt_d) & ok).sum() / ok.sum().clamp_min(1)
                    gate_acc = (((p > 0.5) == fire_d) & own).sum() / own.sum().clamp_min(1)
                stats[0] += loss.item(); stats[1] += hit.item(); stats[2] += gate_acc.item(); stats[3] += 1
        if log:
            print("BC round %2d/%d opp=%d N=%6d | loss %.4f | dest-acc %.3f gate-acc %.3f"
                  % (r + 1, rounds, opp, N, stats[0] / stats[3], stats[1] / stats[3], stats[2] / stats[3]), flush=True)
    save_ckpt(net, BC_CKPT_PATH, meta={"iter": 0, "bc": True})
    print("saved BC init ->", BC_CKPT_PATH)
    return net

if BC_ENABLED:
    bc_net = bc_pretrain()
    if RESUME_FROM is None:      # Run-All: the train cell warm-starts from the BC init
        RESUME_FROM = BC_CKPT_PATH


In [ ]:
net, hist, league = train(total_iters=TOTAL_ITERS, log_every=1, resume_from=RESUME_FROM)

## 14b. Save every league agent as weights

Dumps each league pool member to `CKPT_DIR/league_agents/` as a `save_ckpt` blob (model + config + Elo/label/games). Scripted anchors have no weights, so they are skipped. Reloadable later via `load_snapshot` (e.g. to seed a future run through `LEAGUE_INIT_CKPTS`).

In [ ]:
# Save all league leaderboard agents as weights (run after training).
import os
LEAGUE_OUT = os.path.join(CKPT_DIR, "league_agents")
os.makedirs(LEAGUE_OUT, exist_ok=True)
saved = 0
_lg = globals().get("league") or globals().get("LEAGUE")
if _lg is None:
    raise NameError("No league in scope. Re-run training as `net, hist, league = train(...)` "
                    "with the UPDATED train cell. NOTE: the pool is in-memory only -- a finished "
                    "run whose train() did not return/stash it cannot be recovered.")
for m in sorted(_lg.members, key=lambda mm: -mm["elo"]):
    if m["net"] is None:                       # scripted anchors (random/starter) -> no weights
        continue
    safe = "".join(c if c.isalnum() else "_" for c in m["label"])
    path = os.path.join(LEAGUE_OUT, "%s_elo%04d.pt" % (safe, round(m["elo"])))
    save_ckpt(m["net"], path, meta={"elo": m["elo"], "elo4": m.get("elo4"), "label": m["label"], "games": m["n"], "games4": m.get("n4", 0)})
    saved += 1
    print("saved %-12s elo %6.0f  n=%-4d -> %s" % (m["label"], m["elo"], m["n"], path))
print("saved %d league agents -> %s" % (saved, LEAGUE_OUT))

## 15. Plot the logged curves (return / win-rate)

In [ ]:
# Training-health curves: each metric on its OWN panel + scale (raw = faint, EMA = bold).
def _ema(y, a=0.15):
    out, m = [], None
    for v in y:
        m = float(v) if m is None else (1 - a) * m + a * float(v)
        out.append(m)
    return out

it = hist["iter"]
def _panel(ax, key, title, pct=False, symlog=False):
    y = hist[key]
    ax.plot(it, y, color="tab:blue", alpha=0.25, lw=0.8)
    ax.plot(it, _ema(y), color="tab:blue", lw=1.8)
    ax.set_title(title, fontsize=9); ax.grid(alpha=0.3, lw=0.5)
    if pct: ax.set_ylim(-0.02, 1.02)
    if symlog: ax.set_yscale("symlog", linthresh=1e-3)

fig, ax = plt.subplots(3, 4, figsize=(20, 10))
_panel(ax[0, 0], "return", "episode return (real units)")
_panel(ax[0, 1], "win_rate", "win rate (vs sampled opp)", pct=True)
_panel(ax[0, 2], "learner_elo", "learner Elo (2p)")
_panel(ax[1, 0], "sigma", "exploration (sigma / mean entropy)")
_panel(ax[1, 1], "clipfrac", "PPO clip fraction")
_panel(ax[1, 2], "approx_kl", "approx KL (symlog)", symlog=True)
_panel(ax[0, 3], "grad_norm", "grad norm (pre-clip, symlog)", symlog=True)
_panel(ax[1, 3], "learner_elo4", "learner Elo (4p)")
_panel(ax[2, 3], "share_4p", "share of 4p iterations", pct=True)
_panel(ax[2, 0], "lnch_per_step", "launches / step")
_panel(ax[2, 1], "r_outcome", "reward: outcome channel")
for k, c in (("r_capture", "tab:green"), ("r_launch", "tab:orange"), ("r_milestone", "tab:purple")):
    ax[2, 2].plot(it, hist[k], color=c, alpha=0.20, lw=0.8)
    ax[2, 2].plot(it, _ema(hist[k]), color=c, lw=1.6, label=k.replace("r_", ""))
ax[2, 2].set_title("reward: dense channels"); ax[2, 2].grid(alpha=0.3, lw=0.5); ax[2, 2].legend(fontsize=7)
for a in ax[-1]:
    a.set_xlabel("iter")
fig.suptitle("Orbit Wars v8 -- training health (faint = raw, bold = EMA)", fontsize=12)
plt.tight_layout(); plt.show()

## 16. Optional shape smoke test (default OFF)

A tiny `B=4` dry run of env + policy that checks tensor shapes and runs a couple of steps.
Guarded by `RUN_SMOKE=False` so it does not run heavy work on import / "Run all".

In [ ]:
RUN_SMOKE = False  # set True to run the shape sanity check

if RUN_SMOKE:
    senv = GpuEnv(PLANET_CAP, FLEET_CAP, 8, SHIP_SPEED, DEVICE)
    sworlds = make_world_pool(4, base_seed=123)
    senv.reset(sworlds)
    snet = build_policy()
    ent, em, am, gl = env_encode(senv, 0)
    assert ent.shape == (4, PLANET_CAP, F_DIM), ent.shape
    assert gl.shape == (4, G_DIM), gl.shape
    a_t, logp, value = act(snet, ent, em, am, gl)
    assert a_t.shape == (4, PLANET_CAP, PLANET_CAP + 1), a_t.shape   # alloc (src,dst) | fire gate
    assert logp.shape == (4,) and value.shape == (4,)
    assert torch.isfinite(logp).all(), "log_prob not finite (mask leak?)"
    print("encode/act OK | ent", tuple(ent.shape), "act", tuple(a_t.shape),
          "logp", tuple(logp.shape), "V", tuple(value.shape))
    for opp in (2, 0, 1):   # noop / random / starter
        out = env_step(senv, a_t, opp, None)
        print("step opp=%d | launches=%.0f valid=%.0f captured=%.0f"
              % (opp, out.launches.sum(), out.valid.sum(), out.captured.sum()))
    # self-play step
    o1e, o1m, o1a, o1g = env_encode(senv, 1)
    opp_a, _, _ = act(snet, o1e, o1m, o1a, o1g)
    out = env_step(senv, a_t, 1, opp_a)
    print("self-play step OK | step_ct", senv.step_ct[0].item())
    # v8: 4-player FFA step (scripted + neural seats)
    senv.reset(sworlds, n_players=4)
    ent, em, am, gl = env_encode(senv, 0)
    a_t, _, _ = act(snet, ent, em, am, gl)
    seats = [{"pid": 1, "script": 1}, {"pid": 2, "script": 4}, {"pid": 3, "action": a_t.clone()}]
    out = env_step(senv, a_t, seats=seats)
    s_all, alive_all = settle_n(senv)
    assert s_all.shape == (4, 4) and alive_all.shape == (4, 4), (s_all.shape, alive_all.shape)
    print("4p FFA step OK | p0..p3 ships", [round(float(x)) for x in s_all[0].tolist()])
    print("SMOKE OK")
else:
    print("smoke test disabled (set RUN_SMOKE=True to enable)")